# Rosetta — Recherche d'hyperparamètres et de tokenisation (Optuna, étape 8 révisée)

**Rôle de ce notebook** : mettre en place la recherche d'hyperparamètres Optuna de
l'étape 8 de `docs/plan-seq2seq.md`. Le **tokeniseur entre dans l'espace de
recherche** (il n'est plus fixé par run) et l'**objectif est le SacreBLEU** calculé
en génération libre, plutôt que la cross-entropy de validation. Une couche
d'**adaptateurs** relie ce notebook aux étapes 0→7 du pipeline
(chargement/nettoyage/split, vocabulaire, numérisation, modèle, boucle
d'entraînement) ainsi qu'aux briques de génération et de métrique
(`src/evaluation/`).

**Pourquoi SacreBLEU plutôt que la loss ?** La cross-entropy de validation n'est PAS
comparable entre configs de tokenisation : le nombre de classes du softmax de sortie
diffère d'un facteur ~6 (4 000 pour `bpe4k` contre 23 365 pour `full`), et `max_len`
diffère lui aussi (22 tokens pour `words95`, 32 pour `bpe4k`) -- une loss plus basse
à petit vocabulaire ne signale rien sur la qualité de traduction, seulement un
softmax plus facile. Le SacreBLEU, calculé sur du texte réellement généré puis
détokenisé, est la seule grandeur comparable entre configurations de tokenisation.

**Périmètre tranché** :
- la mécanique Optuna (espace de recherche, pruning, storage SQLite reprenable,
  sauvegarde **incrémentale** après chaque essai, logs, export des résultats,
  figures) est écrite **pour de vrai** ;
- le sous-échantillonnage stratifié par longueur (groupé par cible EN, cohérent
  avec l'anti-fuite de l'étape 3) sert désormais deux usages : un sous-ensemble
  d'environ 20 % du train pour l'entraînement de chaque essai, et un sous-ensemble
  d'environ 2 000 paires de validation pour l'objectif SacreBLEU (la génération
  libre est coûteuse -- hors de question de la faire sur les ~39 000 paires de val
  complètes à chaque epoch de chaque essai) ;
- `vocabSize` est un hyperparamètre numérique tiré par Optuna ; le vocabulaire et les
  DataLoaders correspondants sont construits **dans l'essai, après ce tirage**, sans
  préconstruire ni conserver en mémoire une collection de configurations ;
- les étapes 0→7 (`src/data`, `src/tokenization`, `src/models`, `src/training`)
  sont **branchées** (`src/` expose les 5 symboles attendus) ; ce notebook bascule
  automatiquement sur des **stubs fonctionnels** si ce n'était pas le cas
  (`resolvePipeline()`, tout ou rien).

**Ce que ce notebook NE fait PAS** :
- il n'entraîne pas les modèles finaux (étape 9) ni ne calcule les métriques
  finales par bin de longueur (étape 10) -- il réutilise `src.evaluation.generate`
  et `src.evaluation.metrics` (génération libre, SacreBLEU), écrits pour être
  directement réemployés par ces étapes ultérieures ;
- il ne touche pas à `requirements.txt` (réécrit uniquement par le notebook AED).

**Espace de recherche** : `tokenizationConfig` (catégoriel opaque à 6 valeurs) est
remplacé par `tokenizationMethod` x `vocabSize` (le TPE interpole sur la taille),
`full`/`words95` sortent de l'espace (éliminés par les données), `learningRate` se
resserre sur la zone gagnante, `hiddenDim` s'ouvre jusqu'à 1024 -- détail chiffré en
section 3.

In [1]:
# Paramètres
projectName = "Rosetta"  # nom du projet, utilisé dans les noms de study Optuna
randomSeed = 42  # graine globale (numpy, torch, sampler Optuna) pour reproductibilité

# Langues
srcLang = "fr"  # langue source (colonne du DataFrame)
tgtLang = "en"  # langue cible (colonne du DataFrame)

# Architectures comparées (option A du plan : une étude Optuna séparée par architecture)
architectures = ["rnn", "gru"]  # valeurs passées telles quelles à EncoderStub/DecoderStub

# Attention additive de Bahdanau (étape 9) : variante FIXÉE PAR RUN, PAS un hyperparamètre
# Optuna (elle change l'architecture, pas seulement son réglage). Mettre à True lance une
# étude "avec attention".
useAttention = False

# Embeddings liés (encodeur <-> décodeur <-> projection de sortie, comme MarianMT) :
# variante FIXÉE PAR RUN, PAS un hyperparamètre Optuna. L'espace de recherche (section 3)
# ne tire plus que des configs à vocabulaire CONJOINT (bpe/unigram) -- fr_vocab_size ==
# en_vocab_size est donc toujours vrai, le garde-fou ValueError de build_model ne peut
# plus se déclencher. Gain mesuré (bpe4k/GRU, hiddenDim=256, embDim=128, batch 64, 20%
# du train, 1 epoch) : -56,8% de paramètres (2 644 896 -> 1 141 664) et -9,5% de temps
# par epoch (155,9 -> 141,1 s).
tieEmbeddings = False

# Forcer les stubs même si src/ expose déjà des symboles (utile tant que le pipeline
# est partiellement implémenté : évite de charger le corpus réel avec des budgets
# calibrés pour le corpus jouet).
forceStubs = False

# Longueur & sous-échantillonnage stratifié (étape 8 du plan)
longLengthThreshold = 18  # seuil en mots de la cible EN : strate courte/longue (cf. étape 3)
trainSubsampleFraction = 0.40  # fraction de groupes du TRAIN gardée pour l'entraînement de chaque essai
bleuSubsampleSize = 2000  # taille VISÉE (en paires) du sous-échantillon de VAL pour l'objectif SacreBLEU

# Aucun ensemble de vocabulaires ou de DataLoaders n'est préconstruit. Chaque essai tire
# d'abord `tokenizationMethod` et `vocabSize`, puis construit uniquement ses ressources
# dans `objective(trial)`. Si Optuna retire exactement le même couple méthode/taille,
# SentencePiece peut relire son modèle depuis le cache disque, mais les DataLoaders sont
# recréés pour l'essai et aucun cache inter-essais n'est conservé en mémoire.

# Budget d'entraînement par essai -- mesuré en conditions réelles (words95/GRU et
# bpe4k/GRU, hiddenDim=256, embDim=128, batch 64, sous-échantillon 20% réel du train,
# val COMPLÈTE (~39k paires) pour la loss + génération libre sur ~2000 paires de val
# pour l'objectif SacreBLEU) : ~90-155 s/epoch selon la config ; le surcoût de la
# génération libre est de l'ordre de 1-2% par epoch, négligeable devant le bruit
# inter-run naturel d'une epoch complète.
maxEpochs = 12  # nb max d'epochs par essai (le pruning peut arrêter avant)
batchSize = 64  # taille de batch des DataLoader
patience = 2  # epochs sans amélioration de l'objectif avant arrêt anticipé (hors pruning)
gradClip = 1.0  # norme max pour clip_grad_norm_ (les RNN ont besoin de ce clipping, étape 7)
teacherForcingRatio = 1.0  # ratio de départ (epoch 1) : teacher forcing pur (plan, étape 6)

# Budget Optuna : `tokenizationMethod` x `vocabSize` (section 3) est un espace ORDONNÉ sur
# lequel le TPE interpole, contrairement à un nom de config opaque qui exigerait plusieurs
# essais par config pour discriminer. `full` et `words95` sortent de l'espace -- éliminés
# par les données (section 3). `hiddenDim=1024` coûte cher (récurrence en hidden², x16
# entre 256 et 1024) et le pruning élague sur le SacreBLEU, pas sur la durée -- mitigation :
# durée de chaque essai journalisée (section 5, log compact ET trials_<arch>.csv).
nTrials = 60  # nombre d'essais Optuna par étude

# Espace de recherche (étape 8, cf. section 3) : le tokeniseur entre désormais via
# method x vocabSize plutôt qu'un nom de config opaque.
tokenizationMethodChoices = ["bpe", "unigram"]  # méthode SentencePiece, catégoriel
vocabSizeRange = (2000, 8000)  # borne du vocab_size total FR+EN conjoint
vocabSizeStep = 1  # entier réellement continu -- Optuna peut tirer n'importe quelle taille
# dans vocabSizeRange (ex. 3264), pas seulement des multiples de 1000. `config_name()`
# accepte toute taille entière positive ; chaque taille unique laisse un .model/.vocab
# dans data/processed/tokenizers/ (gitignoré), qui s'accumulent avec des tirages continus.
# Coût par essai (mesuré) : 1,4 s (BPE) à 14,9 s (Unigram) d'entraînement SentencePiece,
# plus la numérisation.
lrRange = (0.0001, 0.005)  # borne du learning rate, log-uniforme, explorant aussi des
# valeurs plus faibles que la zone gagnante observée en pratique (section 3)
hiddenDimChoices = [256, 512, 768, 1024]  # dim. cachée, catégoriel -- 768/1024 ajoutés
# sans mesure préalable (cf. section 3) ; 128 retiré
embDimChoices = [128, 256]  # dimension des embeddings, catégoriel
dropoutRange = (0.0, 0.5)  # borne du dropout
dropoutStep = 0.05  # pas de discrétisation du dropout (suggest_float(..., step=...))
teacherForcingDecayRange = (0.7, 1.0)  # taux de décroissance par epoch ; 1.0 = teacher forcing pur (témoin)
teacherForcingDecayStep = 0.01  # pas de discrétisation du taux

# Pruning (MedianPruner) -- s'appuie maintenant sur le SacreBLEU (direction "maximize")
nStartupTrials = 3  # essais initiaux jamais élagués, le temps d'avoir une médiane fiable
nWarmupSteps = 1  # epochs initiales non évaluées pour le pruning, au sein de chaque essai

# Chemins (relatifs à notebooks/, comme dans l'AED)
reportsDir = "../reports/optuna"  # CSV, JSON et figures produits par ce notebook
storageDir = "../checkpoints"  # storage SQLite Optuna (reprenable d'une exécution à l'autre)
studyStorageName = "optuna_rosetta.db"  # nom du fichier SQLite

# Corpus jouet (stub, désactivé dès que src/ expose les 5 symboles attendus -- c'est
# maintenant le cas : cette valeur n'est plus utilisée qu'en repli, forceStubs=True)
toyCorpusSize = 3000  # nombre de paires FR-EN synthétiques générées (base, avant duplication)

In [2]:
# Imports et configuration globale
# Rendre `src/` importable quel que soit le dossier depuis lequel le kernel est
# lance (Jupyter demarre generalement dans notebooks/). Sans cela,
# resolvePipeline() ne detecterait jamais les modules du pipeline.
import sys
from pathlib import Path

projectRoot = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()),
    Path.cwd(),
)
if str(projectRoot) not in sys.path:
    sys.path.insert(0, str(projectRoot))
import json
import math
import importlib
import time
import warnings
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import optuna
import optuna.visualization.matplotlib as optunaViz
from optuna.importance import PedAnovaImportanceEvaluator

from IPython.display import display

# Génération libre + SacreBLEU : briques génériques indépendantes de la bascule
# stub/réel -- elles s'appuient uniquement sur l'interface `encoder(src)` /
# `decoder.forward_step(...)` commune au modèle réel (`src.models.seq2seq.Seq2Seq`) et
# aux stubs (`Seq2SeqStub`, cf. section 1bis) : aucune branche stub/réel nécessaire ici.
from src.evaluation.generate import generate_translations
from src.evaluation.metrics import corpus_sacrebleu
# Budget cumulatif + échecs non fatals de l'étude Optuna
from src.training.optuna_utils import resolve_tokenization_config, run_study_with_budget

warnings.filterwarnings("ignore")

np.random.seed(randomSeed)
torch.manual_seed(randomSeed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print(f"Projet: {projectName}")
print(
    f"torch: {torch.__version__} | optuna: {optuna.__version__} | "
    f"pandas: {pd.__version__} | numpy: {np.__version__}"
)
print(f"Device: {device}")

c:\Users\emile\Desktop\Projet Plateforme\Rosetta\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Projet: Rosetta
torch: 2.14.0+cpu | optuna: 4.9.0 | pandas: 3.0.5 | numpy: 2.5.2
Device: cpu


## 1. Adaptateurs vers le pipeline (étapes 0→7)

`resolvePipeline()` tente d'importer, depuis `src/`, les symboles attendus pour
chaque étape -- recherchés en **snake_case** (convention `src/` du projet, ex.
`load_splits`), exposés ensuite comme variables locales **camelCase** dans ce
notebook (ex. `loadSplits`). La bascule est **tout ou rien** : si les 5 symboles
ne sont pas TOUS résolus depuis `src/`, le notebook utilise les stubs pour LES
CINQ (sinon un `loadSplits` réel tournerait avec des stubs calibrés pour le
corpus jouet -- incohérent et inutilisable). `generate_translations` et
`corpus_sacrebleu` (section « Imports ») restent EN DEHORS de cette bascule :
ce sont des briques génériques, déjà réelles dans les deux modes, rendues
compatibles avec les stubs par une interface commune (`forward_step`,
`StubVocabulary`, cf. section 1bis ci-dessous).

In [3]:
# Résolution du pipeline : src/ si disponible, sinon bascule sur les stubs (tout ou rien)
def resolvePipeline() -> dict:
    """
    Tente d'importer, depuis `src/`, les symboles réels des étapes 0→7 -- recherchés
    en **snake_case** (convention `src/` du projet : `load_splits`, `build_vocabs`,
    `make_dataloaders`, `build_model`, `train_and_validate`), pas en camelCase.
    Bascule **TOUT OU RIEN** : si les 5 symboles ne sont pas TOUS résolus depuis
    `src/`, le notebook utilise les stubs pour LES CINQ. Sans cette règle, un
    `loadSplits` réel (corpus 264k paires) tournerait avec des stubs calibrés pour
    le corpus jouet (`toyCorpusSize`...) -- incohérent et très lent.

    Affiche un tableau d'état à trois valeurs par symbole :
      - "src/ (utilisé)"                 : résolu, ET les 5 le sont (bascule active)
      - "src/ (disponible, non utilisé)" : résolu, mais au moins un autre symbole manque
      - "stub" / "stub (forcé)"          : non résolu depuis src/, ou `forceStubs=True`
    """
    symbols = [
        ("loadSplits", "load_splits", "src.data.splits", "0-3 (chargement, nettoyage, split)"),
        ("buildVocabs", "build_vocabs", "src.tokenization.vocab_builder", "4 (vocabulaire)"),
        ("makeDataloaders", "make_dataloaders", "src.tokenization.numerize", "5 (numérisation)"),
        ("buildModel", "build_model", "src.models.seq2seq", "6 (encodeur-décodeur)"),
        ("trainAndValidate", "train_and_validate", "src.training.loop", "7 (boucle d'entraînement)"),
    ]

    resolved = {}
    availability = {}

    if not forceStubs:
        for camelName, snakeName, moduleName, _step in symbols:
            try:
                module = importlib.import_module(moduleName)
                resolved[camelName] = getattr(module, snakeName)
                availability[camelName] = True
            except (ImportError, AttributeError):
                availability[camelName] = False

    ready = (not forceStubs) and len(resolved) == len(symbols)

    rows = []
    for camelName, snakeName, moduleName, step in symbols:
        if forceStubs:
            etat = "stub (forcé)"
        elif availability[camelName] and ready:
            etat = "src/ (utilisé)"
        elif availability[camelName]:
            etat = "src/ (disponible, non utilisé)"
        else:
            etat = "stub"
        rows.append({"symbole": snakeName, "module": moduleName, "étape": step, "état": etat})

    display(pd.DataFrame(rows))
    # Tout ou rien : si le pipeline n'est pas prêt, aucun symbole résolu n'est exposé --
    # la cellule suivante (`pipelineInfo["resolved"].get(name, stub)`) retombe alors
    # systématiquement sur le stub, pour LES CINQ symboles, même si certains étaient
    # individuellement résolus depuis src/.
    return {"resolved": resolved if ready else {}, "ready": ready, "rows": rows}


pipelineInfo = resolvePipeline()
pipelineReady = pipelineInfo["ready"]

nPrets = sum(1 for row in pipelineInfo["rows"] if row["état"].startswith("src/"))
nTotal = len(pipelineInfo["rows"])

if not pipelineReady:
    print("=" * 74)
    print("ATTENTION : src/ n'est pas encore branché (étapes 0→7 non implémentées).")
    print("Les résultats ci-dessous viennent d'un CORPUS JOUET synthétique.")
    print("Ils valident la mécanique Optuna (espace, pruning, storage, reprise, logs,")
    print("figures) -- PAS la qualité de traduction d'un vrai pipeline FR->EN.")
    print(f"{nPrets} symbole(s) sur {nTotal} prêt(s) depuis src/ -- résolution TOUT OU RIEN :")
    print("les stubs restent utilisés pour LES CINQ tant qu'il n'y en a pas 5/5.")
    print("La bascule vers src/ se fera automatiquement dès que les 5 le seront.")
    print("=" * 74)
else:
    print("src/ est branché (5/5 symboles) : ce notebook utilise le pipeline réel.")

# Provenance des artefacts : ne jamais mélanger résultats jouet et résultats réels.
runReportsDir = reportsDir if pipelineReady else f"{reportsDir}/stub"
studyNameSuffix = "" if pipelineReady else "-stub"
print(f"Dossier de sortie effectif : {runReportsDir} (suffixe d'étude : '{studyNameSuffix}')")


,symbole,module,étape,état
0,load_splits,src.data.splits,"0-3 (chargement, nettoyage, split)",src/ (utilisé)
1,build_vocabs,src.tokenization.vocab_builder,4 (vocabulaire),src/ (utilisé)
2,make_dataloaders,src.tokenization.numerize,5 (numérisation),src/ (utilisé)
3,build_model,src.models.seq2seq,6 (encodeur-décodeur),src/ (utilisé)
4,train_and_validate,src.training.loop,7 (boucle d'entraînement),src/ (utilisé)


src/ est branché (5/5 symboles) : ce notebook utilise le pipeline réel.
Dossier de sortie effectif : ../reports/optuna (suffixe d'étude : '')


In [4]:
# Stub données (étapes 0→3) : corpus jouet FR→EN avec queue longue et cibles dupliquées
def generateToyCorpus(size: int, seed: int) -> pd.DataFrame:
    """
    Génère un corpus FR→EN synthétique : lexique bijectif bruité, longueurs variables
    avec une vraie queue longue (> longLengthThreshold mots), et cibles EN dupliquées
    (alignements 1→N) via des mots FR synonymes. Remplace fonctionnellement les
    étapes 0→3 tant que `src/data/splits.py` n'existe pas.
    """
    rng = np.random.default_rng(seed)

    vocabWords = 50  # taille du lexique bijectif de base
    frBase = [f"fr{i:02d}" for i in range(vocabWords)]
    enBase = [f"en{i:02d}" for i in range(vocabWords)]
    lexicon = dict(zip(frBase, enBase))  # bijection FR -> EN

    nSynonymPairs = 8  # nb de mots EN atteignables par 2 mots FR différents (1->N légitime)
    synonyms = {f"frsyn{i:02d}": f"en{i:02d}" for i in range(nSynonymPairs)}
    lexicon.update(synonyms)
    frVocabForSentences = frBase + list(synonyms.keys())

    noiseProb = 0.05  # probabilité qu'un mot FR soit traduit "au hasard" (bruit du lexique)

    def translateWord(frWord: str) -> str:
        if rng.random() < noiseProb:
            return str(rng.choice(enBase))
        return lexicon[frWord]

    def sampleLength() -> int:
        if rng.random() < 0.08:  # ~8% de queue longue, garantit des cibles > longLengthThreshold
            return int(rng.integers(longLengthThreshold + 1, longLengthThreshold + 12))
        return int(rng.integers(3, 11))  # cas courant : phrases courtes

    rows = []
    for _ in range(size):
        length = sampleLength()
        frWords = list(rng.choice(frVocabForSentences, size=length))
        enWords = [translateWord(w) for w in frWords]
        rows.append({srcLang: " ".join(frWords), tgtLang: " ".join(enWords)})

    corpus = pd.DataFrame(rows)

    # Duplication de cibles EN (alignement 1->N) : on prend une fraction des paires
    # dont la source contient un mot "base" ayant un synonyme, et on fabrique une
    # source FR alternative (mot remplacé par son synonyme) qui pointe vers LA MÊME
    # cible EN -- exactement le mécanisme "plusieurs sources FR, une cible EN" du
    # vrai corpus (66 542 cibles EN dupliquées dans le vrai corpus).
    dupFraction = 0.15
    hasSynonymBase = corpus[srcLang].apply(
        lambda s: any(f"fr{i:02d}" in s.split() for i in range(nSynonymPairs))
    )
    candidateIdx = corpus.index[hasSynonymBase].to_numpy()
    nDup = int(len(corpus) * dupFraction)
    dupIdx = rng.choice(candidateIdx, size=min(nDup, len(candidateIdx)), replace=False)

    dupRows = []
    for idx in dupIdx:
        frWords = corpus.loc[idx, srcLang].split()
        enTarget = corpus.loc[idx, tgtLang]  # cible EN inchangée : c'est la duplication voulue
        for i in range(nSynonymPairs):
            baseWord = f"fr{i:02d}"
            if baseWord in frWords:
                frWords[frWords.index(baseWord)] = f"frsyn{i:02d}"
                break
        dupRows.append({srcLang: " ".join(frWords), tgtLang: enTarget})

    corpus = pd.concat([corpus, pd.DataFrame(dupRows)], ignore_index=True)
    corpus = corpus.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # mélange
    return corpus


def splitGroupedByTarget(
    df: pd.DataFrame, ratios: tuple[float, float, float], seed: int
) -> dict[str, pd.DataFrame]:
    """
    Split train/val/test groupé par cible EN (anti-fuite, cf. étape 3) : un groupe
    entier (toutes les paires qui partagent la même cible) va dans une seule part.
    Stub simplifié : pas de stratification par longueur ici -- elle est appliquée
    plus loin, sur le train, par `subsampleStratifiedByLength`.
    """
    rng = np.random.default_rng(seed)
    groupKeys = df[tgtLang].unique()
    shuffled = rng.permutation(groupKeys)

    nTrain = int(len(shuffled) * ratios[0])
    nVal = int(len(shuffled) * ratios[1])
    trainKeys = set(shuffled[:nTrain])
    valKeys = set(shuffled[nTrain : nTrain + nVal])
    testKeys = set(shuffled[nTrain + nVal :])

    return {
        "train": df[df[tgtLang].isin(trainKeys)].reset_index(drop=True),
        "val": df[df[tgtLang].isin(valKeys)].reset_index(drop=True),
        "test": df[df[tgtLang].isin(testKeys)].reset_index(drop=True),
    }


def loadSplitsStub(**kwargs) -> dict[str, pd.DataFrame]:
    """Stub des étapes 0→3 : corpus jouet + split groupé par cible EN (80/10/10).

    `**kwargs` (`data_dir`, `processed_dir`, ...) est accepté et IGNORÉ -- présent
    uniquement pour tolérer l'appel réel `loadSplits(data_dir=..., processed_dir=...)`.
    """
    del kwargs
    corpus = generateToyCorpus(toyCorpusSize, randomSeed)
    return splitGroupedByTarget(corpus, ratios=(0.8, 0.1, 0.1), seed=randomSeed)


In [5]:
# Stub vocabulaire / numérisation / DataLoader (étapes 4 et 5)
SPECIAL_TOKENS = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}


class StubVocabulary:
    """Vocabulaire jouet {mot: indice} -- même interface minimale que
    `src.tokenization.vocab_builder.Vocabulary` (`pad_id`/`unk_id`/`sos_id`/`eos_id`,
    `encode`/`decode`/`id_to_piece`/`__len__`) : c'est cette interface commune
    qui permet à `generate_translations` (importée telle quelle depuis `src/`) de
    fonctionner IDENTIQUEMENT en mode stub et en mode réel, sans branche de code
    séparée dans l'objectif Optuna."""

    pad_id = SPECIAL_TOKENS["<pad>"]
    unk_id = SPECIAL_TOKENS["<unk>"]
    sos_id = SPECIAL_TOKENS["<sos>"]
    eos_id = SPECIAL_TOKENS["<eos>"]

    def __init__(self, tokenToId: dict[str, int]) -> None:
        self.tokenToId = tokenToId
        self.idToToken = {i: t for t, i in tokenToId.items()}

    def encode(self, text: str) -> list[int]:
        return [self.tokenToId.get(w, self.unk_id) for w in str(text).split()]

    def decode(self, ids: list[int]) -> str:
        ignore = {self.pad_id, self.sos_id, self.eos_id}
        return " ".join(self.idToToken.get(i, "<unk>") for i in ids if i not in ignore)

    def id_to_piece(self, tokenId: int) -> str:
        return self.idToToken.get(tokenId, "<unk>")

    def __len__(self) -> int:
        return len(self.tokenToId)


def buildVocabFromSeries(series: pd.Series) -> StubVocabulary:
    """Construit un vocabulaire {mot: indice}, tokens spéciaux fixés aux indices 0-3."""
    vocab = dict(SPECIAL_TOKENS)
    counts: dict[str, int] = {}
    for sentence in series:
        for word in sentence.split():
            counts[word] = counts.get(word, 0) + 1
    for word, _ in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
        vocab[word] = len(vocab)
    return StubVocabulary(vocab)


def buildVocabsStub(
    trainDf: pd.DataFrame, config: str | None = None, **kwargs
) -> tuple[StubVocabulary, StubVocabulary]:
    """Stub de l'étape 4 : vocabulaires FR/EN séparés, construits sur le TRAIN seul.

    `config` et `**kwargs` (`processed_dir`, `force_rebuild`, ...) sont acceptés et
    IGNORÉS -- le stub ne connaît qu'un seul vocabulaire mots entiers, quelle que soit
    la config de tokenisation tirée par Optuna (il ne teste que la mécanique, pas
    l'effet réel du tokeniseur) -- présents uniquement pour tolérer l'appel
    réel `buildVocabs(train, tokenizationConfig, processed_dir=...)`.
    """
    del config, kwargs
    frVocab = buildVocabFromSeries(trainDf[srcLang])
    enVocab = buildVocabFromSeries(trainDf[tgtLang])
    return frVocab, enVocab


def numerizeSentence(sentence: str, vocab: StubVocabulary, targetLen: int) -> torch.Tensor:
    """Encode une phrase en indices `[<sos>] + mots + [<eos>]`, paddée/tronquée à targetLen."""
    ids = [vocab.sos_id, *vocab.encode(sentence), vocab.eos_id]
    ids = ids[:targetLen]
    ids = ids + [vocab.pad_id] * (targetLen - len(ids))
    return torch.tensor(ids, dtype=torch.long)


class ToyTranslationDataset(Dataset):
    """Dataset PyTorch pour des paires FR-EN déjà numérisées et paddées à maxLen."""

    def __init__(
        self, df: pd.DataFrame, frVocab: StubVocabulary, enVocab: StubVocabulary, targetLen: int
    ) -> None:
        self.srcTensors = [numerizeSentence(s, frVocab, targetLen) for s in df[srcLang]]
        self.tgtTensors = [numerizeSentence(s, enVocab, targetLen) for s in df[tgtLang]]

    def __len__(self) -> int:
        return len(self.srcTensors)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.srcTensors[idx], self.tgtTensors[idx]


def makeDataloadersStub(
    splits: dict[str, pd.DataFrame],
    frVocab: StubVocabulary,
    enVocab: StubVocabulary,
    targetLen: int | None,
    batchSizeArg: int,
) -> dict[str, DataLoader]:
    """Stub de l'étape 5 : Dataset + DataLoader numérisés et paddés à `targetLen`, un par
    clé de `splits` (accepte n'importe quel ensemble de clés, ex. train/val/valBleu --
    même comportement générique que `make_dataloaders` réel).

    `targetLen=None` -> retombe sur une valeur jouet fixe (16) : le stub ne fait pas la
    dérivation p99 de la vraie implémentation (`src.tokenization.numerize`), il ne fait
    que tolérer l'appel réel `makeDataloaders(comboSplits, frVocab, enVocab, None, batchSize)`.
    """
    if targetLen is None:
        targetLen = 16  # valeur jouet, cf. docstring
    loaders = {}
    for name, df in splits.items():
        dataset = ToyTranslationDataset(df, frVocab, enVocab, targetLen)
        loaders[name] = DataLoader(dataset, batch_size=batchSizeArg, shuffle=(name == "train"))
    return loaders


In [6]:
# Stub modèle (étape 6) : encodeur/décodeur, cellule RNN|GRU paramétrable, teacher forcing
class EncoderStub(nn.Module):
    """Embedding -> cellule récurrente paramétrable (RNN|GRU). Renvoie TOUS les états
    cachés (nécessaire pour greffer l'attention plus tard, cf. étape 9, sans refonte)."""

    def __init__(self, vocabSize: int, embDim: int, hiddenDim: int, cellType: str) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocabSize, embDim, padding_idx=SPECIAL_TOKENS["<pad>"])
        cellCls = {"rnn": nn.RNN, "gru": nn.GRU}[cellType]
        self.rnn = cellCls(embDim, hiddenDim, batch_first=True)

    def forward(self, src: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)  # outputs: (batch, seqLen, hiddenDim)
        return outputs, hidden


class DecoderStub(nn.Module):
    """Embedding -> cellule récurrente -> linéaire. Un pas de décodage à la fois, pour
    permettre le teacher forcing token par token dans `Seq2SeqStub`, ET la génération
    libre pas à pas de `generate_translations` -- même méthode
    `forward_step(input_token, hidden, encoder_outputs=None)` que le vrai `Decoder`
    (`encoder_outputs` accepté et ignoré : le stub ne modélise pas l'attention)."""

    def __init__(
        self, vocabSize: int, embDim: int, hiddenDim: int, cellType: str, dropout: float
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocabSize, embDim, padding_idx=SPECIAL_TOKENS["<pad>"])
        self.dropout = nn.Dropout(dropout)
        cellCls = {"rnn": nn.RNN, "gru": nn.GRU}[cellType]
        self.rnn = cellCls(embDim, hiddenDim, batch_first=True)
        self.out = nn.Linear(hiddenDim, vocabSize)

    def forward_step(
        self,
        inputToken: torch.Tensor,
        hidden: torch.Tensor,
        encoder_outputs: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        del encoder_outputs  # stub sans attention (cf. docstring de la classe)
        embedded = self.dropout(self.embedding(inputToken))
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.out(output.squeeze(1))
        return prediction, hidden


class Seq2SeqStub(nn.Module):
    """Ossature encodeur-décodeur (pas d'attention ici, cf. étape 9) avec
    `teacherForcingRatio` intégré : 1.0 = teacher forcing pur, 0.0 = génération libre."""

    def __init__(
        self,
        frVocabSize: int,
        enVocabSize: int,
        embDim: int,
        hiddenDim: int,
        cellType: str,
        dropout: float,
    ) -> None:
        super().__init__()
        self.encoder = EncoderStub(frVocabSize, embDim, hiddenDim, cellType)
        self.decoder = DecoderStub(enVocabSize, embDim, hiddenDim, cellType, dropout)
        self.enVocabSize = enVocabSize

    def forward(
        self, src: torch.Tensor, tgt: torch.Tensor, teacherForcingRatioArg: float
    ) -> torch.Tensor:
        batchSizeLocal, tgtLenSeq = tgt.shape
        tensorDevice = src.device
        encoderOutputs, hidden = self.encoder(src)

        outputs = torch.zeros(batchSizeLocal, tgtLenSeq, self.enVocabSize, device=tensorDevice)
        inputToken = tgt[:, 0:1]  # <sos>
        for t in range(1, tgtLenSeq):
            prediction, hidden = self.decoder.forward_step(inputToken, hidden, encoderOutputs)
            outputs[:, t, :] = prediction
            useTeacherForcing = torch.rand(1).item() < teacherForcingRatioArg
            top1 = prediction.argmax(1, keepdim=True)
            inputToken = tgt[:, t : t + 1] if useTeacherForcing else top1
        return outputs


def buildModelStub(
    frVocabSize: int,
    enVocabSize: int,
    embDim: int,
    hiddenDim: int,
    cellType: str,
    dropout: float,
    use_attention: bool = False,
    tie_embeddings: bool = False,
) -> Seq2SeqStub:
    """Stub de l'étape 6 : construit un `Seq2SeqStub` (RNN ou GRU selon `cellType`).

    `use_attention` et `tie_embeddings` sont acceptés et IGNORÉS (le stub ne modélise
    ni l'attention (cf. étape 9) ni les embeddings liés) -- présents uniquement pour
    tolérer l'appel réel `buildModel(..., use_attention=useAttention, tie_embeddings=tieEmbeddings)`.
    """
    del use_attention, tie_embeddings
    return Seq2SeqStub(frVocabSize, enVocabSize, embDim, hiddenDim, cellType, dropout)


In [7]:
# Stub boucle d'entraînement (étape 7) : loss ignorant le pad, clipping, scheduled
# sampling, objectif optionnel piloté par une métrique externe -- même contrat que
# `src.training.loop.train_and_validate` (voir sa docstring pour le détail).
def _scheduledTeacherForcingRatioStub(epoch: int, start: float, decay: float) -> float:
    """Décroissance exponentielle, indépendante de `maxEpochs` (cf. `src/training/loop.py`)."""
    return start * (decay ** (epoch - 1))


def trainAndValidateStub(
    model: nn.Module,
    loaders: dict[str, DataLoader],
    maxEpochsArg: int,
    lr: float,
    gradClipArg: float,
    teacherForcingRatioArg: float,
    patienceArg: int,
    deviceArg: torch.device,
    on_epoch_end: Callable[[int, float], bool] | None = None,
    pad_id: int = 0,
    checkpoint_path: str | None = None,
    teacher_forcing_decay: float = 1.0,
    objective_fn: Callable[[nn.Module], float] | None = None,
    direction: str = "minimize",
) -> dict:
    """
    Stub de l'étape 7 -- même contrat que `train_and_validate` réel : scheduled
    sampling (`teacher_forcing_decay`), objectif optionnel (`objective_fn`, remplace la
    val loss pour piloter `on_epoch_end`/l'early stopping), `direction`
    ("minimize"/"maximize"). `on_epoch_end(epoch, metricValue)` est le point d'accroche
    du pruning Optuna (qui peut aussi interrompre en levant une exception).
    """
    del checkpoint_path  # le stub n'écrit jamais de checkpoint (accepté pour tolérer l'appel réel)
    if direction not in ("minimize", "maximize"):
        raise ValueError(f"direction doit être 'minimize' ou 'maximize', reçu {direction!r}")

    model.to(deviceArg)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history: dict[str, list] = {
        "trainLoss": [], "valLoss": [], "valPerplexity": [], "teacherForcingRatio": [],
    }
    bestValLoss = float("inf")
    bestMetric = float("inf") if direction == "minimize" else float("-inf")
    epochsWithoutImprovement = 0

    for epoch in range(1, maxEpochsArg + 1):
        epochTeacherForcingRatio = _scheduledTeacherForcingRatioStub(
            epoch, teacherForcingRatioArg, teacher_forcing_decay
        )

        model.train()
        trainLossSum, nTrainBatches = 0.0, 0
        for src, tgt in loaders["train"]:
            src, tgt = src.to(deviceArg), tgt.to(deviceArg)
            optimizer.zero_grad()
            output = model(src, tgt, epochTeacherForcingRatio)
            outputDim = output.shape[-1]
            loss = criterion(output[:, 1:, :].reshape(-1, outputDim), tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gradClipArg)
            optimizer.step()
            trainLossSum += loss.item()
            nTrainBatches += 1
        trainLoss = trainLossSum / max(nTrainBatches, 1)

        model.eval()
        valLossSum, nValBatches = 0.0, 0
        with torch.no_grad():
            for src, tgt in loaders["val"]:
                src, tgt = src.to(deviceArg), tgt.to(deviceArg)
                output = model(src, tgt, 0.0)  # pas de triche en validation
                outputDim = output.shape[-1]
                loss = criterion(output[:, 1:, :].reshape(-1, outputDim), tgt[:, 1:].reshape(-1))
                valLossSum += loss.item()
                nValBatches += 1
        valLoss = valLossSum / max(nValBatches, 1)
        valPerplexity = math.exp(min(valLoss, 20)) if math.isfinite(valLoss) else float("inf")

        history["trainLoss"].append(trainLoss)
        history["valLoss"].append(valLoss)
        history["valPerplexity"].append(valPerplexity)
        history["teacherForcingRatio"].append(epochTeacherForcingRatio)

        bestValLoss = min(bestValLoss, valLoss)

        metricValue = objective_fn(model) if objective_fn is not None else valLoss
        improved = (
            metricValue < bestMetric - 1e-4 if direction == "minimize" else metricValue > bestMetric + 1e-4
        )

        if improved:
            bestMetric = metricValue
            epochsWithoutImprovement = 0
        else:
            epochsWithoutImprovement += 1

        shouldStop = bool(on_epoch_end(epoch, metricValue)) if on_epoch_end is not None else False
        if shouldStop or epochsWithoutImprovement >= patienceArg:
            break

    return {"history": history, "bestValLoss": bestValLoss, "bestObjective": bestMetric}


In [8]:
# Sélection effective des fonctions (src/ si disponible, sinon stub) et chargement des données
loadSplits = pipelineInfo["resolved"].get("loadSplits", loadSplitsStub)
buildVocabs = pipelineInfo["resolved"].get("buildVocabs", buildVocabsStub)
makeDataloaders = pipelineInfo["resolved"].get("makeDataloaders", makeDataloadersStub)
buildModel = pipelineInfo["resolved"].get("buildModel", buildModelStub)
trainAndValidate = pipelineInfo["resolved"].get("trainAndValidate", trainAndValidateStub)

splits = loadSplits(data_dir="../data", processed_dir="../data/processed")
print(
    f"Split -- train: {len(splits['train'])} paires, val: {len(splits['val'])} paires, "
    f"test: {len(splits['test'])} paires"
)

# La méthode et la taille sont des hyperparamètres Optuna. `configName()` ne sélectionne
# pas une bibliothèque préconstruite : il produit uniquement le nom canonique utilisé
# pour construire le vocabulaire demandé après le tirage du trial.
if pipelineReady:
    from src.tokenization.vocab_builder import config_name

    configName = config_name
else:

    def configName(method: str, vocabSize: int) -> str:
        """Équivalent stub de `config_name()` -- même convention, mode jouet uniquement."""
        return f"{method}{vocabSize // 1000}k"

print(
    f"Espace tokenisation Optuna : méthodes={tokenizationMethodChoices}, "
    f"vocabSize={vocabSizeRange} (entier continu, pas {vocabSizeStep})."
)


[etape 3] cache trouvé dans ..\data\processed, relecture (force_rebuild=False).
Split -- train: 183994 paires, val: 39367 paires, test: 39517 paires
Espace tokenisation Optuna : méthodes=['bpe', 'unigram'], vocabSize=(2000, 8000) (entier continu, pas 1, lot K).


## 2. Sous-échantillonnage stratifié par longueur

Étape 8 du plan, réutilisé deux fois :
1. un sous-ensemble d'environ 20 % du **train**, pour entraîner chaque essai
   rapidement, **sans perdre les phrases longues** (c'est sur elles que l'écart
   RNN/GRU se joue) ;
2. un sous-ensemble d'environ 2 000 paires de **val**, pour l'objectif SacreBLEU
   (la génération libre est coûteuse -- hors de question de la faire sur les
   ~39 000 paires de val complètes à chaque epoch de chaque essai).

Le regroupement se fait par **cible EN** (jamais couper un groupe), cohérent avec
l'anti-fuite de l'étape 3 : un groupe dupliqué (alignement 1→N) reste entier dans
le sous-ensemble ou en dehors, jamais réparti des deux côtés.


In [9]:
# Sous-échantillonnage stratifié par longueur, groupé par cible EN -- réutilisé pour le
# train (entraînement de chaque essai) ET pour la val (objectif SacreBLEU, section 3bis)
def subsampleStratifiedByLength(
    df: pd.DataFrame, fraction: float, threshold: int, seed: int
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Sous-échantillonne `df` en conservant la proportion de phrases longues. Regroupe
    par cible EN (jamais couper un groupe en deux, cohérent avec l'anti-fuite de
    l'étape 3), tire aléatoirement des GROUPES entiers dans chaque strate (longueur
    de groupe = nb de mots de sa cible EN, seuil `threshold`), puis reconstitue les
    paires. Renvoie (sous-ensemble, diagnostic avant/après).
    """
    rng = np.random.default_rng(seed)
    uniqueTargets = df[tgtLang].unique()
    lengthByTarget = {t: len(t.split()) for t in uniqueTargets}

    shortKeys = np.array([t for t in uniqueTargets if lengthByTarget[t] <= threshold], dtype=object)
    longKeys = np.array([t for t in uniqueTargets if lengthByTarget[t] > threshold], dtype=object)

    def sampleGroupKeys(keys: np.ndarray) -> np.ndarray:
        if len(keys) == 0:
            return keys
        nKeep = min(max(1, int(round(len(keys) * fraction))), len(keys))
        return rng.choice(keys, size=nKeep, replace=False)

    keptKeys = set(sampleGroupKeys(shortKeys)) | set(sampleGroupKeys(longKeys))
    subset = df[df[tgtLang].isin(keptKeys)].reset_index(drop=True)

    def diagnosticRow(label: str, frame: pd.DataFrame) -> dict:
        keys = frame[tgtLang].unique()
        nLongues = sum(1 for k in keys if lengthByTarget[k] > threshold)
        nGroupes = len(keys)
        return {
            "jeu": label,
            "nPaires": len(frame),
            "nGroupes": nGroupes,
            "nLongues": nLongues,
            "proportionLongues": nLongues / nGroupes if nGroupes else 0.0,
        }

    diagnostic = pd.DataFrame([diagnosticRow("avant", df), diagnosticRow("après", subset)])
    return subset, diagnostic


In [10]:
# Sous-échantillonnage du TRAIN (~20%, indépendant de la config de tokenisation -- ce
# sont des phrases FR/EN brutes, pas encore numérisées) + calcul de la fraction visant
# ~bleuSubsampleSize paires pour le sous-échantillon VAL de l'objectif SacreBLEU.
trainSubsample, subsampleDiag = subsampleStratifiedByLength(
    splits["train"], trainSubsampleFraction, longLengthThreshold, randomSeed
)
display(subsampleDiag)

nLonguesApres = int(subsampleDiag.loc[subsampleDiag["jeu"] == "après", "nLongues"].iloc[0])
assert nLonguesApres > 0, (
    "Le sous-ensemble stratifié ne contient aucune phrase longue -- "
    "vérifier longLengthThreshold ou le corpus jouet."
)

propAvant = float(subsampleDiag.loc[subsampleDiag["jeu"] == "avant", "proportionLongues"].iloc[0])
propApres = float(subsampleDiag.loc[subsampleDiag["jeu"] == "après", "proportionLongues"].iloc[0])
print(
    f"Proportion de groupes longs (train) -- avant: {propAvant:.1%}, après: {propApres:.1%} "
    f"(écart {abs(propApres - propAvant):.1%})"
)
print(f"Sous-ensemble retenu pour l'entraînement: {len(trainSubsample)} paires ({nLonguesApres} groupes longs).")

bleuSubsampleFraction = min(1.0, bleuSubsampleSize / max(len(splits["val"]), 1))
print(
    f"Fraction visée pour le sous-échantillon val de l'objectif SacreBLEU: "
    f"{bleuSubsampleFraction:.4f} (cible ~{bleuSubsampleSize} paires sur {len(splits['val'])})"
)


# Le sous-échantillon de validation est indépendant des hyperparamètres : on le tire une
# fois sur le texte brut. Les vocabulaires et les DataLoaders, eux, sont construits dans
# chaque essai après que la méthode et la taille ont été choisies par Optuna.
valBleuSubsample, _diagBleu = subsampleStratifiedByLength(
    splits["val"], bleuSubsampleFraction, longLengthThreshold, randomSeed
)


def buildTrialResources(tokenizationMethod: str, vocabSize: int) -> dict:
    """Construit uniquement les ressources demandées par un essai Optuna.

    L'appel intervient dans `objective(trial)`, après `suggestHyperparams`. Il n'existe
    donc aucune bibliothèque de vocabulaires préconstruite à parcourir. SentencePiece
    peut relire un modèle disque si le même couple est tiré plusieurs fois, tandis que
    les DataLoaders sont systématiquement recréés pour isoler chaque essai.
    """
    config = configName(tokenizationMethod, vocabSize)
    debutConstruction = time.perf_counter()
    frVocab, enVocab = buildVocabs(
        splits["train"], config, processed_dir="../data/processed", force_rebuild=False
    )
    comboSplits = {
        "train": trainSubsample,
        "val": splits["val"],
        "valBleu": valBleuSubsample,
    }
    loadersConfig = makeDataloaders(comboSplits, frVocab, enVocab, None, batchSize)

    return {
        "config": config,
        "frVocab": frVocab,
        "enVocab": enVocab,
        "loaders": loadersConfig,
        "references": list(valBleuSubsample[tgtLang]),
        "constructionSeconds": time.perf_counter() - debutConstruction,
    }


print(
    "Ressources de tokenisation : aucune préconstruction ; méthode et taille seront "
    "tirées puis construites au début de chaque essai Optuna."
)


,jeu,nPaires,nGroupes,nLongues,proportionLongues
0,avant,183994,138026,1790,0.012969
1,après,36875,27605,358,0.012969


Proportion de groupes longs (train) -- avant: 1.3%, après: 1.3% (écart 0.0%)
Sous-ensemble retenu pour l'entraînement: 36875 paires (358 groupes longs).
Fraction visée pour le sous-échantillon val de l'objectif SacreBLEU: 0.0508 (cible ~2000 paires sur 39367)
Ressources de tokenisation : aucune préconstruction ; méthode et taille seront tirées puis construites au début de chaque essai Optuna.


## 3. Espace de recherche

Les essais antérieurs (86 essais, 24 complets, `checkpoints/optuna_rosetta.db`)
tranchent l'espace de recherche :

| config | essais complets | meilleur SacreBLEU | médiane |
|---|---|---|---|
| `unigram4k` | 4 | **11,36** | 10,89 |
| `bpe4k` | 5 | 11,09 | 9,92 |
| `bpe8k` | 6 | 8,54 | 0,99 |
| `unigram8k` | 3 | 6,11 | 0,33 |
| `words95` | 6 | 0,88 | 0,46 |
| `full` | 0 | -- | -- |

1. **`words95` est éliminé par les données**, pas par intuition : 0,88 de SacreBLEU
   contre 11,36 pour `unigram4k` (facteur ~13).
2. **`full` n'a aucun essai complet** sur 86 essais et coûte le plus cher par epoch
   (softmax de sortie à 23 365 classes) -- retiré plutôt que de continuer à
   consommer du budget sans jamais finir.
3. **L'optimum observé est à 4k, et 8k est déjà en retrait** (11,36/11,09 à 4k contre
   8,54/6,11 à 8k) -- la zone prometteuse est plus probablement EN DESSOUS de 4k
   qu'AU-DESSUS de 8k.

Conséquence : `tokenizationConfig` (catégoriel opaque à 6 valeurs) est remplacé par
le couple `tokenizationMethod` (`bpe`/`unigram`, catégoriel) x `vocabSize`
(2 000 -> 8 000, **entier réellement continu** -- ex. 3264 est une taille valable) --
résolu en nom de config par `configName()` (mode réel : `config_name()` de
`src.tokenization.vocab_builder`, source de vérité UNIQUE de la convention de nom,
jamais réimplémentée en double, accepte désormais toute taille entière positive). Le
TPE **interpole** désormais sur la taille du vocabulaire au lieu de traiter 6 noms de
config sans relation entre eux -- il n'a plus besoin d'échantillonner plusieurs
essais PAR config pour discriminer. Chaque essai réentraîne SentencePiece (1,4 s pour
BPE, jusqu'à 14,9 s pour Unigram, mesuré) et renumérise sur SA taille exacte -- coût
assumé et négligeable devant celui d'un essai complet (dizaines de secondes à
plusieurs minutes). Revers assumé : les tailles continues se recoupent rarement d'un
essai à l'autre, donc chaque taille UNIQUE laisse un `.model`/`.vocab` dans
`data/processed/tokenizers/` (gitignoré), qui s'accumulent. `full` et `words95`
restent implémentés dans `TOKENIZATION_CONFIGS` (essais antérieurs interprétables)
mais **sortent** de l'espace de recherche.

Reste de l'espace :
- `learningRate` : plage resserrée à **(0,001 -> 0,005)**, log-uniforme -- les
  meilleurs essais antérieurs se concentrent dans cette zone (le learning rate reste
  l'hyperparamètre le plus déterminant, ~80 % de l'effet selon le plan).
- `hiddenDim` : `[256, 512, 768, 1024]` -- 768 et 1024 ajoutés **sans mesure
  préalable** (réserve non résolue) ; 128 retiré (jamais le meilleur essai
  antérieur). Le coût de récurrence d'un GRU croît en hidden² (x16 entre 256 et
  1024), et le pruning élague sur le SacreBLEU, pas la durée -- un essai
  `hiddenDim=1024` lent mais prometteur peut monopoliser le budget. Mitigation
  retenue : la durée de CHAQUE essai est désormais journalisée (log compact ET
  `trials_<arch>.csv`, section 5), pour que le phénomène reste visible et
  corrigeable -- pas de garde-fou de durée. Point le plus lourd attendu de
  l'espace : `vocabSize=2000` x `hiddenDim=1024` (un petit vocabulaire allonge les
  séquences, et la longueur de séquence pèse plus que la taille du vocabulaire sur
  ce corpus).
- `embDim`, `dropout`, `teacherForcingDecay` : inchangés.

**Conséquence pour `tieEmbeddings`** : toutes les configs de cet espace sont à
vocabulaire CONJOINT (`bpe`/`unigram`), donc `fr_vocab_size == en_vocab_size` est
désormais TOUJOURS vrai -- le garde-fou `ValueError` de `build_model` ne peut plus se
déclencher. `tieEmbeddings` redevient réglable (cellule Paramètres, toujours FIXÉ PAR
RUN, hors espace Optuna) : gain mesuré (`bpe4k`/GRU) de -56,8 % de paramètres et
-9,5 % de temps par epoch.

`cellType` (RNN/GRU) reste **hors** de cet espace (option A du plan : deux études
séparées, comparaison « meilleur plafond vs meilleur plafond »).

In [11]:
# Espace de recherche des hyperparamètres (cf. section 3)
def suggestHyperparams(trial: optuna.Trial) -> dict:
    """Échantillonne un jeu d'hyperparamètres pour un essai Optuna. `tokenizationMethod`
    x `vocabSize` remplacent un nom de config opaque -- le TPE interpole désormais sur
    la taille du vocabulaire (n'importe quel entier dans vocabSizeRange, pas seulement
    des multiples de 1000), résolue en nom de config par `configName()` (section 1bis,
    même convention que `config_name()` de `src.tokenization.vocab_builder`, qui
    accepte lui aussi toute taille positive)."""
    tokenizationMethod = trial.suggest_categorical("tokenizationMethod", tokenizationMethodChoices)
    vocabSize = trial.suggest_int("vocabSize", *vocabSizeRange, step=vocabSizeStep)
    return {
        "tokenizationMethod": tokenizationMethod,
        "vocabSize": vocabSize,
        "tokenizationConfig": configName(tokenizationMethod, vocabSize),
        "learningRate": trial.suggest_float("learningRate", *lrRange, log=True),
        "hiddenDim": trial.suggest_categorical("hiddenDim", hiddenDimChoices),
        "embDim": trial.suggest_categorical("embDim", embDimChoices),
        "dropout": trial.suggest_float("dropout", *dropoutRange, step=dropoutStep),
        "teacherForcingDecay": trial.suggest_float(
            "teacherForcingDecay", *teacherForcingDecayRange, step=teacherForcingDecayStep
        ),
    }

## 4. Fonction objectif et pruning (objectif SacreBLEU)

`makeObjective(cellType, ...)` construit une closure `objective(trial)` qui : tire la
méthode et la taille du vocabulaire avec les autres hyperparamètres, construit ensuite
uniquement les ressources de cet essai (`buildTrialResources`), entraîne le modèle et
calcule le SacreBLEU en **génération libre** (`generate_translations`, jamais de teacher
forcing) sur le sous-échantillon val (~2 000 paires) à CHAQUE epoch (`objective_fn`),
le rapporte à Optuna (`trial.report`) et lève `optuna.TrialPruned` si
`trial.should_prune()` ou si la métrique devient NaN/inf (essai raté, à couper
immédiatement). L'étude est `direction="maximize"` (SacreBLEU, pas la val loss).


In [12]:
# Fonction objectif Optuna (fermeture sur l'architecture) : objectif SacreBLEU
def makeObjective(cellType: str, deviceArg: torch.device) -> Callable[[optuna.Trial], float]:
    """
    Construit la fonction objectif Optuna pour l'architecture `cellType`. Renvoie le
    meilleur SacreBLEU atteint (direction "maximize") -- remplace la val loss,
    non comparable entre configs de tokenisation (cf. cellule titre). En FIN d'essai,
    génère et affiche aussi quelques traductions en GÉNÉRATION LIBRE (pas de
    teacher forcing), stockées dans `trial.user_attrs["exemplesTraduction"]`.
    """

    def objective(trial: optuna.Trial) -> float:
        hp = suggestHyperparams(trial)
        torch.manual_seed(randomSeed + trial.number)  # reproductible mais différent par essai

        # Construction strictement par essai : Optuna choisit d'abord la taille comme
        # n'importe quel autre hyperparamètre, puis seul ce vocabulaire est matérialisé.
        resources = buildTrialResources(hp["tokenizationMethod"], hp["vocabSize"])
        trial.set_user_attr("tokenizationConfig", resources["config"])
        trial.set_user_attr("resourceConstructionSeconds", resources["constructionSeconds"])
        frVocab, enVocab = resources["frVocab"], resources["enVocab"]
        loadersConfig = resources["loaders"]
        references = resources["references"]
        # Taille RÉELLEMENT obtenue : SentencePiece tourne avec
        # hard_vocab_limit=False et peut ne pas atteindre exactement hp["vocabSize"] --
        # la taille DEMANDÉE est déjà journalisée par Optuna (colonne params_vocabSize
        # de trials_dataframe()), on n'enregistre ici QUE l'obtenue, pour ne jamais
        # analyser un chiffre faux (cf. vocab_builder.build_vocabs, qui logge les deux).
        trial.set_user_attr("vocabSizeObtained", len(frVocab))

        model = buildModel(
            len(frVocab),
            len(enVocab),
            hp["embDim"],
            hp["hiddenDim"],
            cellType,
            hp["dropout"],
            use_attention=useAttention,
            tie_embeddings=tieEmbeddings,
        )

        def objectiveFn(m: nn.Module) -> float:
            hypotheses = generate_translations(m, loadersConfig["valBleu"], enVocab, deviceArg)
            return corpus_sacrebleu(hypotheses, references)

        def onEpochEnd(epoch: int, metricValue: float) -> bool:
            if not math.isfinite(metricValue):
                raise optuna.TrialPruned(f"objectif non fini à l'epoch {epoch}")
            trial.report(metricValue, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
            return False

        result = trainAndValidate(
            model,
            {"train": loadersConfig["train"], "val": loadersConfig["val"]},
            maxEpochs,
            hp["learningRate"],
            gradClip,
            teacherForcingRatio,
            patience,
            deviceArg,
            on_epoch_end=onEpochEnd,
            teacher_forcing_decay=hp["teacherForcingDecay"],
            objective_fn=objectiveFn,
            direction="maximize",
        )

        bestObjective = result["bestObjective"]
        if not math.isfinite(bestObjective):
            raise optuna.TrialPruned("objectif final non fini")

        # Quelques traductions en GÉNÉRATION LIBRE, en fin d'essai -- coût
        # négligeable (~1,0-1,5 s sur les ~2000 paires de valBleu, cf. cellule
        # Paramètres). À NE PAS CONFONDRE avec le round-trip encode/decode du
        # tokeniseur (vocab_builder.py, étape 4, sans modèle ni teacher forcing) :
        # ceci est une VRAIE traduction, décodage glouton autorégressif, qui ne lit
        # jamais la cible réelle (cf. src.evaluation.generate.generate_translations).
        nExamples = min(3, len(references))
        exampleHypotheses = generate_translations(model, loadersConfig["valBleu"], enVocab, deviceArg)
        exemples = [
            {
                "source": str(valBleuSubsample[srcLang].iloc[i]),
                "hypothese": exampleHypotheses[i],
                "reference": references[i],
            }
            for i in range(nExamples)
        ]
        print(f"  [essai {trial.number}] exemples de traduction (génération libre):")
        for exemple in exemples:
            print(f"    FR:  {exemple['source']!r}")
            print(f"    HYP: {exemple['hypothese']!r}")
            print(f"    REF: {exemple['reference']!r}")
        trial.set_user_attr("exemplesTraduction", exemples)

        return bestObjective

    return objective

## 4bis. Cycle de vie du vocabulaire par essai

Aucun vocabulaire n'est construit avant l'étude. Dans chaque `objective(trial)`, Optuna
choisit d'abord `tokenizationMethod` et `vocabSize`, puis `buildTrialResources` crée le
vocabulaire et les DataLoaders nécessaires à ce seul essai. Le temps de préparation est
stocké dans `trial.user_attrs["resourceConstructionSeconds"]` afin de rester visible
séparément du score SacreBLEU.

Le cache disque de SentencePiece reste autorisé : si Optuna tire de nouveau exactement
le même couple méthode/taille, le modèle de tokenisation est rechargé au lieu d'être
réentraîné. Il n'existe toutefois plus de préconstruction globale ni de cache de
DataLoaders entre les essais.


In [13]:
# Contrôle de cohérence de l'hyperparamètre numérique, sans construire de vocabulaire.
assert vocabSizeRange[0] > 0 and vocabSizeRange[0] <= vocabSizeRange[1]
assert vocabSizeStep >= 1  # 1 = entier réellement continu
print(
    f"vocabSize sera tiré par Optuna entre {vocabSizeRange[0]} et {vocabSizeRange[1]} "
    f"(pas {vocabSizeStep}, entier continu), puis construit dans l'essai."
)


vocabSize sera tiré par Optuna entre 2000 et 8000 (pas 1, entier continu), puis construit dans l'essai.


## 5. Lancement des études (option A : une étude par architecture)

`runStudy(cellType)` crée (ou reprend, `load_if_exists=True`) une étude Optuna
`direction="maximize"` par architecture, stockée en SQLite
(`checkpoints/optuna_rosetta.db`) pour pouvoir interrompre puis reprendre la
recherche sans perdre les essais déjà faits. Le callback
`makeIncrementalSaveCallback` réécrit `trials_<arch>.csv` et
`best_params_<arch>.json` **après CHAQUE essai** (pas en fin d'étude) : une
interruption ne doit jamais perdre les essais déjà terminés.

**Budget cumulatif et échecs non fatals** (`src.training.optuna_utils.run_study_with_budget`,
extrait dans `src/` pour être testable, `tests/test_optuna_utils.py`) :

- **`nTrials` devient une CIBLE TOTALE**, pas un nombre d'essais additionnels --
  `remaining_trials` compte les essais `COMPLETE`/`PRUNED`/`FAIL` déjà enregistrés
  (tous ont consommé du calcul) et ne lance que le complément. Si le budget est
  déjà atteint, aucun essai n'est lancé -- message explicite, ventilation par état.
- **Un essai en échec n'interrompt plus l'étude** : l'objectif est enveloppé
  (`wrap_objective_with_failure_logging`), toute exception AUTRE que
  `optuna.TrialPruned` est journalisée (trace complète, en append) dans
  `failures_<arch>.log`, puis relevée -- Optuna marque l'essai `FAIL` et,
  grâce à `catch=(Exception,)` dans `study.optimize`, l'étude CONTINUE.
  `optuna.TrialPruned` n'est JAMAIS intercepté par ce mécanisme : c'est le
  fonctionnement normal du pruning, pas un échec.
- Les artefacts CSV/JSON (`writeStudyArtifacts`) sont réécrits inconditionnellement
  après `runStudy`, même si le budget était déjà atteint (aucun nouvel essai) --
  une reprise produit toujours des fichiers à jour.

In [14]:
# Lancement (ou reprise) d'une étude Optuna pour une architecture donnée, avec
# sauvegarde incrémentale (après CHAQUE essai, pas en fin d'étude) et budget
# cumulatif + échecs non fatals (cf. src/training/optuna_utils.py -- run_study_with_budget)
def writeStudyArtifacts(study: optuna.Study, cellType: str, reportsDirArg: str) -> None:
    """Réécrit `trials_<arch>.csv` et `best_params_<arch>.json` à partir de l'état
    ACTUEL de `study` -- appelée après CHAQUE essai (callback, ci-dessous) ET une
    fois de plus après `runStudy` même si aucun essai n'a été lancé (budget déjà
    atteint) : une reprise doit produire des fichiers à jour même sans nouveau
    calcul."""
    Path(reportsDirArg).mkdir(parents=True, exist_ok=True)
    trialsDf = study.trials_dataframe()
    if "duration" in trialsDf.columns:
        # Colonne dérivée, lisible en secondes : rend visible un essai lent
        # (ex. hiddenDim=1024) qui monopolise le budget alors que le pruning élague
        # sur le SacreBLEU, pas sur la durée (cf. section 3).
        trialsDf["durationSeconds"] = trialsDf["duration"].dt.total_seconds()
    trialsDf.to_csv(Path(reportsDirArg) / f"trials_{cellType}.csv", index=False)

    completedTrials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if completedTrials:  # best_params n'a de sens qu'avec au moins un essai COMPLETE
        record = {
            "cellType": cellType,
            "bestObjective": study.best_value,
            "bestParams": study.best_params,
            "nTrialsCompleted": len(completedTrials),
            "nTrialsTotal": len(study.trials),
            "pipelineReady": pipelineReady,
        }
        with open(Path(reportsDirArg) / f"best_params_{cellType}.json", "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)


def makeIncrementalSaveCallback(cellType: str, reportsDirArg: str) -> Callable[[optuna.Study, optuna.trial.FrozenTrial], None]:
    """Callback Optuna : log compact + réécrit les artefacts (`writeStudyArtifacts`)
    après CHAQUE essai -- une interruption ne perd donc jamais les essais déjà
    terminés."""

    def callback(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
        value = trial.value if trial.value is not None else float("nan")
        # Durée de CHAQUE essai : le pruning élague sur le SacreBLEU, pas sur la
        # durée -- un essai hiddenDim=1024 lent mais prometteur peut monopoliser
        # le budget (réserve, cf. section 3) ; visible ici.
        dureeSecondes = trial.duration.total_seconds() if trial.duration is not None else float("nan")
        print(
            f"  [essai {trial.number:>3}] état={trial.state.name:<9} valeur={value:.4f} "
            f"durée={dureeSecondes:6.1f}s params={trial.params}"
        )
        writeStudyArtifacts(study, cellType, reportsDirArg)

    return callback


def runStudy(cellType: str, deviceArg: torch.device) -> optuna.Study:
    """
    Lance (ou reprend) l'étude Optuna pour l'architecture `cellType`.

    Le nom d'étude inclut `studyNameSuffix` ("-stub" en mode jouet, "" en mode réel) :
    sans ce suffixe, `load_if_exists=True` ferait reprendre une exécution réelle par
    l'étude qui contient déjà les essais du corpus jouet, et `trials_dataframe()` /
    `best_params` mélangeraient silencieusement deux corpus différents.

    `nTrials` est une CIBLE TOTALE (pas un nombre d'essais additionnels, cf.
    `run_study_with_budget`) -- reprendre une étude qui contient déjà `nTrials`
    essais ou plus n'en relance aucun. Un essai qui plante n'interrompt plus
    l'étude : il est marqué `FAIL`, sa trace complète est journalisée dans
    `failures_<cellType>.log`, et l'étude continue.
    """
    Path(storageDir).mkdir(parents=True, exist_ok=True)
    storageUrl = f"sqlite:///{storageDir}/{studyStorageName}"
    sampler = optuna.samplers.TPESampler(seed=randomSeed)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=nStartupTrials, n_warmup_steps=nWarmupSteps)

    study = optuna.create_study(
        # tokenizationMethod x vocabSize remplacent tokenizationConfig (catégoriel
        # opaque à 6 valeurs), hiddenDim s'étend à 768/1024, learningRate se
        # resserre à (0.001, 0.005) -- cf. section 3. Toute modification de
        # l'espace DOIT incrémenter la version de ce nom d'étude, sinon une
        # reprise mélangerait deux espaces différents dans la même étude.
        study_name=f"rosetta-{cellType}-v4{studyNameSuffix}",
        storage=storageUrl,
        load_if_exists=True,
        direction="maximize",  # SacreBLEU, pas la val loss
        sampler=sampler,
        pruner=pruner,
    )

    objective = makeObjective(cellType, deviceArg)
    failuresLogPath = Path(runReportsDir) / f"failures_{cellType}.log"
    run_study_with_budget(
        study,
        objective,
        n_trials=nTrials,
        log_path=failuresLogPath,
        callbacks=[makeIncrementalSaveCallback(cellType, runReportsDir)],
    )
    # Réécrit les artefacts même si aucun essai n'a été lancé (budget déjà
    # atteint) -- une reprise doit produire des fichiers à jour sans nouveau calcul.
    writeStudyArtifacts(study, cellType, runReportsDir)
    return study

In [15]:
# Boucle sur les architectures (option A : une étude complète par architecture)
optuna.logging.set_verbosity(optuna.logging.WARNING)

studies: dict[str, optuna.Study] = {}
for cellType in architectures:
    print(f"\n=== Étude Optuna -- architecture: {cellType} ===")
    studies[cellType] = runStudy(cellType, device)



=== Étude Optuna -- architecture: rnn ===
Étude 'rosetta-rnn-v3' -- 40 essai(s) déjà comptabilisé(s) dans le budget sur 40 visé(s) au total (COMPLETE=8, PRUNED=32). Budget déjà atteint, 0 essai à lancer -- augmente n_trials pour en lancer davantage.

=== Étude Optuna -- architecture: gru ===
Étude 'rosetta-gru-v3' -- 40 essai(s) déjà comptabilisé(s) dans le budget sur 40 visé(s) au total (COMPLETE=9, FAIL=1, PRUNED=31). Budget déjà atteint, 0 essai à lancer -- augmente n_trials pour en lancer davantage.


## 6. Résultats et points de contrôle

`trials_<arch>.csv` et `best_params_<arch>.json` sont déjà à jour (écrits après
CHAQUE essai, cf. section 5, sauvegarde incrémentale). Cette section construit le
tableau comparatif RNN vs GRU et les figures de diagnostic Optuna (historique
d'optimisation, importance des hyperparamètres -- désormais 7, dont
`tokenizationMethod` et `vocabSize` (qui remplacent `tokenizationConfig`) --,
coordonnées parallèles) via le backend matplotlib d'Optuna
(`plotly` n'est pas installé).


In [16]:
# Tableau comparatif RNN vs GRU -- les fichiers par architecture sont déjà à jour
# (écrits après chaque essai par le callback de la section 5)
comparisonRows = []
for cellType, study in studies.items():
    completedTrials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    prunedTrials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    print(f"{cellType}: meilleur SacreBLEU = {study.best_value:.2f} -> best_params_{cellType}.json")

    comparisonRows.append(
        {
            "cellType": cellType,
            "bestObjective": study.best_value,
            **study.best_params,
            "nCompleted": len(completedTrials),
            "nPruned": len(prunedTrials),
        }
    )

comparisonDf = pd.DataFrame(comparisonRows)
comparisonPath = Path(runReportsDir) / "comparaison_studies.csv"
comparisonDf.to_csv(comparisonPath, index=False)
display(comparisonDf)
print(f"Tableau comparatif RNN vs GRU -> {comparisonPath}")


rnn: meilleur SacreBLEU = 1.05 -> best_params_rnn.json
gru: meilleur SacreBLEU = 12.46 -> best_params_gru.json


,cellType,bestObjective,tokenizationMethod,vocabSize,learningRate,hiddenDim,embDim,dropout,teacherForcingDecay,nCompleted,nPruned
0,rnn,1.047387,unigram,6169,0.002647,1024,256,0.3,0.87,8,32
1,gru,12.457704,unigram,6663,0.002424,1024,256,0.0,1.00,9,31


Tableau comparatif RNN vs GRU -> ..\reports\optuna\comparaison_studies.csv


In [17]:
# Figures de diagnostic Optuna (backend matplotlib, PedAnova pour l'importance)
for cellType, study in studies.items():
    try:
        ax = optunaViz.plot_optimization_history(study)
        ax.figure.savefig(Path(runReportsDir) / f"fig_history_{cellType}.png", bbox_inches="tight")
        plt.close(ax.figure)
    except Exception as exc:  # noqa: BLE001 -- on veut continuer même si une figure échoue
        print(f"[avertissement] figure d'historique indisponible pour {cellType}: {exc}")

    try:
        evaluator = PedAnovaImportanceEvaluator()  # pur numpy, pas besoin de scikit-learn
        ax = optunaViz.plot_param_importances(study, evaluator=evaluator)
        ax.figure.savefig(Path(runReportsDir) / f"fig_importance_{cellType}.png", bbox_inches="tight")
        plt.close(ax.figure)
    except Exception as exc:  # noqa: BLE001
        print(f"[avertissement] figure d'importance indisponible pour {cellType}: {exc}")

    try:
        ax = optunaViz.plot_parallel_coordinate(study)
        ax.figure.savefig(Path(runReportsDir) / f"fig_parallel_{cellType}.png", bbox_inches="tight")
        plt.close(ax.figure)
    except Exception as exc:  # noqa: BLE001
        print(f"[avertissement] figure des coordonnées parallèles indisponible pour {cellType}: {exc}")


In [18]:
# Checklist -- points de contrôle de l'étape 8 du plan (révisée)
print("=" * 74)
print("Checklist -- points de contrôle de l'étape 8 du plan (révisée)")
print("=" * 74)

checks = []

# 1. Le sous-ensemble 20 % contient bien des phrases longues.
checks.append(("Le sous-ensemble 20 % (train) contient des phrases longues", nLonguesApres > 0))

# 2. Tous les essais sont loggés (trials_dataframe non vide, une ligne par essai).
allTrialsLogged = all(
    len(studies[c].trials_dataframe()) == len(studies[c].trials) and len(studies[c].trials) > 0
    for c in architectures
)
checks.append(("Tous les essais sont loggés (trials_dataframe)", allTrialsLogged))

# 3. Meilleurs hyperparamètres récupérés par architecture (fichiers JSON présents,
#    écrits incrémentalement après chaque essai, cf. section 5).
bestParamsExported = all(
    (Path(runReportsDir) / f"best_params_{c}.json").exists() for c in architectures
)
checks.append(("Meilleurs hyperparamètres exportés par architecture (incrémental)", bestParamsExported))

for label, ok in checks:
    print(f"[{'OK' if ok else 'KO'}] {label}")

assert all(ok for _, ok in checks), "Au moins un point de contrôle de l'étape 8 a échoué."
print("\nTous les points de contrôle de l'étape 8 (révisée) sont validés.")


Checklist -- points de contrôle de l'étape 8 du plan (révisée, lot F1)
[OK] Le sous-ensemble 20 % (train) contient des phrases longues
[OK] Tous les essais sont loggés (trials_dataframe)
[OK] Meilleurs hyperparamètres exportés par architecture (incrémental)

Tous les points de contrôle de l'étape 8 (révisée) sont validés.


## 7. Limites et suite

- **20 % ≠ optimum du dataset complet** : les hyperparamètres réglés sur le
  sous-ensemble stratifié sont un bon point de départ, pas l'optimum exact du
  dataset complet -- léger risque de sur-régularisation (dropout, notamment).
- **Le tokeniseur est désormais DANS l'espace de recherche** : conséquence directe,
  l'objectif passe de la cross-entropy au SacreBLEU (cf. cellule titre). La méthode
  et la taille sont tirées avant de construire le vocabulaire et les DataLoaders
  propres à chaque essai ; aucun cache de ressources inter-essais ni aucune
  préconstruction globale n'est conservé.
- **Coût de la génération libre** : le surcoût mesuré par rapport à l'ancienne
  validation en teacher forcing seule est non négligeable -- d'où le
  sous-échantillon val (~2 000 paires stratifiées) pour l'objectif, plutôt que les
  ~39 000 paires de val complètes.
- **`full` et `words95` sortent de l'espace de recherche** : éliminés par les
  essais antérieurs (BLEU 0,88 et aucun essai complet contre 11,36 pour
  `unigram4k`, cf. section 3) -- ils restent implémentés dans
  `TOKENIZATION_CONFIGS` pour que ces essais restent interprétables, mais Optuna
  ne les tire plus jamais.
- **`tieEmbeddings` redevient réglable** : cet espace ne tire plus que des configs
  à vocabulaire CONJOINT (`bpe`/`unigram`), le garde-fou `ValueError` de
  `build_model` ne peut donc plus se déclencher -- reste FIXÉ PAR RUN (comme
  `useAttention`), hors espace Optuna (gain mesuré : -56,8 % de paramètres,
  -9,5 % de temps par epoch).
- **`hiddenDim=1024` sans mesure préalable** : le pruning élague sur le SacreBLEU,
  pas sur la durée -- mitigation retenue, PAS un garde-fou : la durée de chaque
  essai est journalisée (log compact ET `trials_<arch>.csv`, section 5) pour
  rendre le phénomène visible et corrigeable.
- **Brancher `src/` ne demande aucune modification de ce notebook** : `resolvePipeline()`
  détecte les 5 symboles attendus et bascule seule (`pipelineReady`).
  `generate_translations` et `corpus_sacrebleu` sont importés directement
  depuis `src/` dans les deux modes -- les stubs (`Seq2SeqStub`, `StubVocabulary`)
  exposent la même interface minimale que les objets réels pour rester compatibles
  sans branche de code séparée.
- **Séparation de provenance jouet/réel** : dossier `reports/optuna/stub/` et
  suffixe d'étude `-stub` tant que `pipelineReady` est faux, bascule automatique
  vers `reports/optuna/` et les études réelles sinon. `forceStubs` force le mode
  jouet même si `src/` est déjà branché.
- **Budget de ce run** : `nTrials` argumenté par la mesure réelle (cf. cellule
  Paramètres), pas un smoke-test arbitraire.

## 8. Étape 9 — entraînements finaux

Trois entraînements sur le corpus **complet** (183 994 paires train, 39 367 val),
**reprenables** (`src.training.final.train_final_model`) :

1. **RNN** — meilleurs hyperparamètres de l'étude `rosetta-rnn` (`best_params_rnn.json`).
2. **GRU** — meilleurs hyperparamètres de l'étude `rosetta-gru` (`best_params_gru.json`).
3. **GRU + attention de Bahdanau** — **mêmes hyperparamètres que le GRU seul (2)**,
   seul `use_attention=True` change. Il n'existe pas d'étude Optuna dédiée à
   l'attention (les études Optuna sont par `cellType`, pas par variante
   d'attention, cf. `docs/plan-seq2seq.md`) — réutiliser les hyperparamètres du
   GRU rend la comparaison « avec/sans attention » interprétable : **une seule
   variable change**.

Si `best_params_<arch>.json` est absent (étude Optuna pas encore lancée), des
valeurs par défaut raisonnables sont utilisées à la place — annoncé
clairement, jamais de plantage silencieux.

Chaque run gère lui-même sa **reprise** (`state.json`, `checkpoint.pt`,
`best.pt`, `history.json` sous `reports/runs/<run_id>/`) : relancer cette
section après une interruption reprend exactement où elle s'est arrêtée.

In [19]:
# Paramètres -- étape 9 (entraînements finaux)
optunaParamsDir = "../reports/optuna"  # best_params_<arch>.json produits par la section 5
runsDir = "../reports/runs"  # reports/runs/<run_id>/ -- state.json, checkpoint.pt, best.pt, history.json

finalRunTag = "v2"  # tag du dossier de run, sous reports/runs/<finalRunTag>/<cellType>/


def runDirFor(cellType: str, tag: str | None = None) -> str:
    """Chemin du dossier de run pour `cellType`, sous le tag `tag` (par defaut
    `finalRunTag`). `tag=""` pointe sur les runs hérités, à la racine de `runsDir`
    (reports/runs/<cellType>/)."""
    tagEffectif = finalRunTag if tag is None else tag
    return str(Path(runsDir) / tagEffectif / cellType) if tagEffectif else str(Path(runsDir) / cellType)


finalArchitectures = ["rnn", "gru", "gru_attention"]  # les 3 runs de l'étape 9
tokenizationConfigDefault = "unigram4k"  # repli si best_params_<arch>.json est absent, OU
# si son format est ambigu (résolution impossible, cf. resolve_tokenization_config) --
# unigram4k est le meilleur essai antérieur (SacreBLEU 11,36, médiane 10,89 sur 4 essais
# complets, section 3) ; l'ancien défaut words95 est 13x moins bon (0,88) et hors de
# l'espace de recherche, il n'a plus aucun sens ici.

# Budget d'entraînement -- coût mesuré : le padding dynamique divise la boucle
# d'entraînement par ~2 (mesuré sur le sous-échantillon 20% : 73,4 -> 36,4 s/epoch).
# UNE epoch réelle sur le corpus COMPLET (train 183 994 + val 39 367), config
# words95/GRU, hiddenDim=256, embDim=128, batch=64, dropout=0.3 : **203,0 s/epoch**.
# Extrapolation aux 2 autres runs (même corpus/hyperparamètres, seule l'architecture
# change) via les ratios mesurés séparément :
#   - RNN : ratio RNN/GRU mesuré sur le sous-échantillon 20% (73,1 s / 96,7 s ≈ 0,756)
#     -> RNN ≈ 203,0 * 0,756 ≈ 153,5 s/epoch
#   - GRU+attention : ratio avec/sans attention mesuré (8000 train + 2000 val, mêmes
#     hyperparamètres : 26,6 s / 16,0 s ≈ 1,66) -> GRU+attention ≈ 203,0 * 1,66 ≈ 337,0 s/epoch
# Pire cas sans early stopping, budgets desserrés (finalMaxEpochs=50) :
# RNN ≈ 153,5*50 ≈ 2h08, GRU ≈ 203,0*50 ≈ 2h49, GRU+attention ≈ 337,0*50 ≈ 4h41
# -> total pire cas ≈ 9h38 pour les 3 runs. Avec early stopping (finalPatience=8), le
# pire cas ci-dessus reste la seule borne fiable tant que les 3 runs n'ont pas tourné.
finalMaxEpochs = 50  # budget max par run -- l'early stopping (patience ci-dessous) arrête généralement avant
# `finalPatience` compte des epochs SANS amélioration du SacreBLEU (objective_fn de
# train_final_model, cellule 32), PAS de la val loss -- corrige un désaccord de métrique
# avec la recherche Optuna (section 4) : celle-ci sélectionne déjà ses hyperparamètres en
# maximisant ce même SacreBLEU en génération libre (cellule 16), alors que la val loss en
# génération libre devient pathologique pour les hyperparamètres qu'elle retient
# (teacherForcingDecay proche de 1.0, dropout faible, donc quasi aucun scheduled sampling en
# entraînement) : la CE explose dès que la séquence générée s'écarte de la référence, alors
# même que le modèle continue de produire de MEILLEURES traductions (le BLEU, lui, continue
# de s'améliorer). Piloter best.pt/l'early stopping sur la val loss dans ce cas retiendrait
# systématiquement un modèle sous-entraîné -- constaté sur un run gru : valLoss = [5.51,
# 5.76, 5.92, 6.22, 6.26], croissante dès l'epoch 1, meilleure epoch = 1 avec patience=8. La
# val loss et la perplexité restent calculées et journalisées (history.json) -- seule la
# DÉCISION (best.pt, early stopping) change de métrique.
finalPatience = 8  # epochs sans amélioration du BLEU (objective_fn, cf. commentaire ci-dessus)
finalTeacherForcingStart = 1.0  # teacher forcing pur à l'epoch 1, décroît au taux teacherForcingDecay

defaultHyperparams = {
    # Valeurs par défaut si best_params_<arch>.json est absent (Optuna pas encore
    # lancé) -- un point de départ raisonnable, PAS une optimisation.
    "tokenizationConfig": tokenizationConfigDefault,
    "learningRate": 1e-3,
    "hiddenDim": 256,
    "embDim": 128,
    "dropout": 0.3,
    "teacherForcingDecay": 0.9,
}

# --- Overrides de VALIDATION uniquement (JAMAIS activés dans le fichier livré) ---
# `debugMode=True` réduit drastiquement les budgets (epochs, sous-échantillon
# train, taille du test, taille de l'échantillon BERTScore) pour vérifier que
# le notebook tourne de bout en bout en quelques minutes -- sert UNIQUEMENT à
# valider la mécanique (chargement best_params, reprise, génération,
# métriques), jamais à produire de vrais résultats de comparaison.
debugMode = False
debugTrainSubsampleFraction = 0.01  # ~1840 paires train si debugMode=True
debugMaxEpochs = 2
debugTestSampleSize = 100
debugBertscoreSampleSize = 20

In [20]:
# Imports -- étapes 9 et 10
from src.evaluation.generate import generate_translations_resumable
from src.evaluation.metrics import corpus_meteor, evaluate_by_length_bin
from src.evaluation.report import metric_disagreements, sample_predictions, teacher_forcing_bias
from src.training.final import load_state, mark_step_done, params_differences, train_final_model

In [21]:
# Chargement des meilleurs hyperparamètres (Optuna, section 5) -- si absent,
# valeurs par défaut + message clair (pas de plantage : l'étude Optuna
# correspondante n'a peut-être pas encore été lancée).
#
# `record["bestParams"]` ne contient QUE ce que Optuna a réellement suggéré
# (`tokenizationMethod`/`vocabSize`, cf. section 3) -- JAMAIS `tokenizationConfig`,
# qui n'est qu'une clé DÉRIVÉE par `suggestHyperparams`, pas un paramètre Optuna
# suggéré. `resolve_tokenization_config` (`src.training.optuna_utils`) garantit la
# clé, sans repli muet -- un format déjà au format `tokenizationConfig` reste
# lisible tel quel, sinon il est dérivé par `configName()`.
def loadBestParams(cellType: str) -> dict:
    chemin = Path(optunaParamsDir) / f"best_params_{cellType}.json"
    if chemin.exists():
        with open(chemin, encoding="utf-8") as f:
            record = json.load(f)
        print(f"[etape 9] hyperparamètres '{cellType}' chargés depuis {chemin} (bestObjective={record.get('bestObjective')}).")
        resolved = resolve_tokenization_config(
            dict(record["bestParams"]), default=tokenizationConfigDefault, cell_type=cellType, source=chemin
        )
        print(f"[etape 9] '{cellType}' : tokenizationConfig='{resolved['tokenizationConfig']}'.")
        return resolved
    print(
        f"[etape 9] ATTENTION : {chemin} introuvable (étude Optuna '{cellType}' pas encore lancée) -- "
        f"utilisation des valeurs par défaut : {defaultHyperparams}."
    )
    return dict(defaultHyperparams)


hyperparamsRnn = loadBestParams("rnn")
hyperparamsGru = loadBestParams("gru")
# GRU + attention réutilise DÉLIBÉRÉMENT les hyperparamètres du GRU seul (cf. cellule titre)
hyperparamsGruAttention = dict(hyperparamsGru)

runHyperparams = {"rnn": hyperparamsRnn, "gru": hyperparamsGru, "gru_attention": hyperparamsGruAttention}
runUseAttention = {"rnn": False, "gru": False, "gru_attention": True}
runCellType = {"rnn": "rnn", "gru": "gru", "gru_attention": "gru"}  # gru_attention EST un GRU (+ attention)

[etape 9] hyperparamètres 'rnn' chargés depuis ..\reports\optuna\best_params_rnn.json (bestObjective=1.0473868722543425).
[etape 9] 'rnn' : tokenizationConfig='unigram6169'.
[etape 9] hyperparamètres 'gru' chargés depuis ..\reports\optuna\best_params_gru.json (bestObjective=12.457704491657038).
[etape 9] 'gru' : tokenizationConfig='unigram6663'.


In [22]:
# Corpus complet + vocabulaires/DataLoaders par config de tokenisation retenue
# (chaque run utilise la config choisie par SA propre étude Optuna --
# tokenizationConfig fait partie de l'espace de recherche -- gru_attention
# réutilise TOUJOURS celle du GRU). Cache par config (au plus 2 configs
# distinctes en pratique ici, rnn et gru pouvant différer) -- pas de politique
# d'éviction, contrairement au notebook Optuna : au plus 2 jeux de tenseurs
# simultanés à conserver, pas 6.
splitsFinal = loadSplits(data_dir="../data", processed_dir="../data/processed")
print(
    f"[etape 9] corpus complet -- train={len(splitsFinal['train'])} "
    f"val={len(splitsFinal['val'])} test={len(splitsFinal['test'])}"
)

# Sous-échantillon VAL stratifié pour l'objectif BLEU de l'étape 9 -- MÊME mécanique que
# la section 4bis (cellules 11-12, subsampleStratifiedByLength, même seed randomSeed, même
# taille visée bleuSubsampleSize) : la recherche Optuna (section 4) sélectionne déjà ses
# hyperparamètres en maximisant le SacreBLEU en génération libre -- garder le même objectif
# ici (cellule 32) évite de désaccorder les deux étapes (cf. cellule Paramètres, cellule 28).
finalBleuSubsampleFraction = min(1.0, bleuSubsampleSize / max(len(splitsFinal["val"]), 1))
valBleuSubsampleFinal, _diagBleuFinal = subsampleStratifiedByLength(
    splitsFinal["val"], finalBleuSubsampleFraction, longLengthThreshold, randomSeed
)
finalBleuReferences = list(valBleuSubsampleFinal[tgtLang])
print(
    f"[etape 9] sous-échantillon val pour l'objectif BLEU : {len(valBleuSubsampleFinal)} paires "
    f"(cible ~{bleuSubsampleSize} sur {len(splitsFinal['val'])})."
)

_finalResourcesCache: dict[str, dict] = {}


def getFinalResources(tokenizationConfig: str) -> dict:
    if tokenizationConfig in _finalResourcesCache:
        return _finalResourcesCache[tokenizationConfig]

    trainDf = splitsFinal["train"]
    if debugMode:
        trainDf, _diag = subsampleStratifiedByLength(trainDf, debugTrainSubsampleFraction, longLengthThreshold, randomSeed)
        print(f"[etape 9] DEBUG : train sous-échantillonné à {len(trainDf)} paires (config='{tokenizationConfig}').")

    frVocab, enVocab = buildVocabs(trainDf, tokenizationConfig, processed_dir="../data/processed")
    loadersConfig = makeDataloaders(
        {"train": trainDf, "val": splitsFinal["val"], "valBleu": valBleuSubsampleFinal},
        frVocab, enVocab, None, batchSize,
    )
    # `max_len` dérivé du TRAIN par `makeDataloaders` ci-dessus -- récupéré ici pour être
    # réutilisé tel quel lors de la construction ULTÉRIEURE du DataLoader de test (étape 10) :
    # `makeDataloaders({"test": ...}, ..., max_len=None, ...)` échouerait (dérivation
    # automatique exige une clé "train" dans le dict passé, absente pour un DataLoader de
    # test isolé) -- voir `TranslationDataset.max_len`.
    maxLenConfig = loadersConfig["train"].dataset.max_len

    resources = {"frVocab": frVocab, "enVocab": enVocab, "loaders": loadersConfig, "maxLen": maxLenConfig}
    _finalResourcesCache[tokenizationConfig] = resources
    return resources

[etape 3] cache trouvé dans ..\data\processed, relecture (force_rebuild=False).
[etape 9] corpus complet -- train=183994 val=39367 test=39517


In [ ]:
# --- Entraînements finaux (3 runs), reprenables ---------------------------------
finalResults: dict[str, dict] = {}
finalRunDirs: dict[str, str] = {}

for cellType in finalArchitectures:
    hp = runHyperparams[cellType]
    # Accès DIRECT (pas de .get(..., default)) : tokenizationConfig est garanti présent,
    # résolu une fois pour toutes par resolve_tokenization_config dans loadBestParams.
    tokenizationConfig = hp["tokenizationConfig"]
    resources = getFinalResources(tokenizationConfig)
    frVocab, enVocab = resources["frVocab"], resources["enVocab"]
    loadersRun = resources["loaders"]

    useAttentionRun = runUseAttention[cellType]
    cellTypeRun = runCellType[cellType]

    def modelFactory(hp=hp, frVocab=frVocab, enVocab=enVocab, useAttentionRun=useAttentionRun, cellTypeRun=cellTypeRun):
        return buildModel(
            len(frVocab), len(enVocab), hp["embDim"], hp["hiddenDim"], cellTypeRun, hp["dropout"],
            use_attention=useAttentionRun,
        )

    # Objectif SacreBLEU en génération libre sur le sous-échantillon val de l'étape 9
    # (valBleuSubsampleFinal / finalBleuReferences, cellule 31) -- MÊME mécanique que la
    # recherche Optuna (cellule 16 : generate_translations + corpus_sacrebleu), pas de
    # réimplémentation. `direction="maximize"` fait piloter best.pt/l'early stopping par ce
    # BLEU plutôt que par la val loss en génération libre (cf. cellule Paramètres, cellule 28)
    # -- val loss et perplexité restent calculées/journalisées dans history.json (garde-fou
    # `train_final_model`), seule la décision change.
    def objectiveFn(
        m: nn.Module, enVocab=enVocab, valBleuLoader=loadersRun["valBleu"], references=finalBleuReferences
    ) -> float:
        hypotheses = generate_translations(m, valBleuLoader, enVocab, device)
        return corpus_sacrebleu(hypotheses, references)

    runDir = runDirFor(cellType)  # reports/runs/<finalRunTag>/<cellType>/
    finalRunDirs[cellType] = runDir
    maxEpochsRun = debugMaxEpochs if debugMode else finalMaxEpochs

    print(
        f"\n=== Étape 9 -- run '{cellType}' (config='{tokenizationConfig}', "
        f"useAttention={useAttentionRun}, maxEpochs={maxEpochsRun}) ==="
    )
    resultatRun = train_final_model(
        run_id=cellType,
        model_factory=modelFactory,
        loaders={"train": loadersRun["train"], "val": loadersRun["val"]},
        params=hp,
        run_dir=runDir,
        max_epochs=maxEpochsRun,
        device=device,
        grad_clip=gradClip,
        patience=finalPatience,
        teacher_forcing_ratio=finalTeacherForcingStart,
        seed=randomSeed,
        objective_fn=objectiveFn,
        direction="maximize",
    )
    finalResults[cellType] = resultatRun
    print(
        f"[etape 9] run '{cellType}' terminé -- bestObjective(BLEU)={resultatRun['bestObjective']:.4f} "
        f"bestValLoss={resultatRun['bestValLoss']:.4f} "
        f"(resumed={resultatRun['resumed']}, startEpoch={resultatRun['startEpoch']})."
    )

[etape 4] vocab unigram6169/joint (FR+EN): modèle SentencePiece en cache -> ..\data\processed\tokenizers\unigram6169_joint.model
[etape 4] config=unigram6169: vocab FR=6169, vocab EN=6169 (vocabulaire CONJOINT FR+EN, même objet (fr_vocab is en_vocab))
[etape 4] config=unigram6169: vocab_size demandé=6169, obtenu=6169
[etape 4] round-trip FR (exemple): "lorsqu' il a demandé qui avait cassé la fenêtre tous les garçons ont pris un air innocent" -> "lorsqu' il a demandé qui avait cassé la fenêtre tous les garçons ont pris un air innocent"
[etape 5] max_len dérivé automatiquement (maximum observé sur le train + bornes <sos>/<eos>, garde-fou): 87
[etape 5] train: max_len=87 (garde-fou, padding dynamique par batch), 0/367988 séquences tronquées (0.00%)
[etape 5] val: max_len=87 (garde-fou, padding dynamique par batch), 0/78734 séquences tronquées (0.00%)

=== Étape 9 -- run 'rnn' (config='unigram6169', useAttention=False, maxEpochs=50) ===
[etape 9] run 'rnn' epoch 1/50: trainLoss=5.2864 valL

In [ ]:
# Résumé des 3 entraînements finaux
resumeFinalDf = pd.DataFrame(
    [
        {
            "run": cellType,
            "tokenizationConfig": runHyperparams[cellType]["tokenizationConfig"],
            "useAttention": runUseAttention[cellType],
            "bestValLoss": finalResults[cellType]["bestValLoss"],
            "nEpochs": len(finalResults[cellType]["history"]["trainLoss"]),
            "resumed": finalResults[cellType]["resumed"],
        }
        for cellType in finalArchitectures
    ]
)
display(resumeFinalDf)

## 9. Étape 10 — évaluation

Génération **libre** (jamais en teacher forcing, cf. `src/evaluation/generate.py`)
sur le test, **incrémentale** (reprise automatique via `hypotheses.jsonl`) :

- **SacreBLEU** (`lowercase=True`, corpus normalisé en minuscules) + **METEOR** sur
  le **test complet**.
- **BERTScore** (distilbert, léger) sur un **échantillon stratifié ~3000 paires**
  (coût CPU) -- même échantillon pour les 3 runs.
- Ventilation **courtes (≤ 18 mots) / longues (> 18 mots)**, mesurée sur la
  référence EN (`count_words`, cohérent avec le split de l'étape 3).
- 3 analyses : biais du teacher forcing (Q14a), désaccords entre métriques,
  exemples source/hypothèse/référence (Q14d).

In [ ]:
# Paramètres -- étape 10 (évaluation)
# Coûts MESURÉS (poids non entraînés pour la génération/teacher_forcing_bias, la
# durée ne dépend que de l'archi/taille, pas de la qualité) :
#   - génération libre sur les 39 517 paires de test : RNN 14,7 s | GRU 16,7 s |
#     GRU+attention 32,1 s (total 3 modèles : ~1 min)
#   - SacreBLEU sur 39 517 paires : ~1,1 s/modèle
#   - METEOR sur 5 000 paires (mesure) extrapolé à 39 517 : ~14 s/modèle
#   - BERTScore (distilbert) sur 3 000 paires, modèle déjà en cache : ~9,8 s/modèle
#   - teacher_forcing_bias sur le val complet (39 367 paires, 2 forward/batch) : ~36 s/modèle
# Total étape 10 pour les 3 modèles : de l'ordre de 4-5 minutes, NÉGLIGEABLE
# devant le budget de l'étape 9 (heures).
lengthBinThreshold = 18  # seuil (mots, référence EN) -- cohérent avec l'étape 3 du split
bertscoreSampleSize = 3000  # taille VISÉE de l'échantillon stratifié pour BERTScore (coût CPU)
bertscoreModelType = "distilbert-base-uncased"
samplePredictionsN = 20  # nb d'exemples source/hypothèse/référence affichés (Q14d)
disagreementsReferenceRun = "gru"  # run utilisé pour l'analyse des désaccords entre métriques

In [ ]:
# Test complet (ou réduit en DEBUG) + helper de rechargement du meilleur modèle
testDfFinal = splitsFinal["test"]
if debugMode:
    testDfFinal = testDfFinal.iloc[:debugTestSampleSize].reset_index(drop=True)
    print(f"[etape 10] DEBUG : test réduit à {len(testDfFinal)} paires.")

sourcesFinal = list(testDfFinal["fr"])
referencesFinal = list(testDfFinal["en"])


def loadBestModel(cellType: str):
    """Reconstruit vocabulaire + modèle depuis SON PROPRE run : les hyperparamètres
    viennent de `state.json` du dossier de run (`runDirFor(cellType)`, tag
    `finalRunTag`), JAMAIS de `runHyperparams` (best_params COURANTS, section 5,
    chargés en cellule "Chargement des meilleurs hyperparamètres") -- sinon la
    reconstruction casse dès qu'une étude Optuna plus récente a tourné (mismatch
    de dimensions au chargement de `best.pt`). `runHyperparams` ne sert ICI qu'à
    comparer et avertir en cas de divergence, jamais à construire le modèle.
    """
    runDir = runDirFor(cellType)
    state = load_state(runDir)
    runParams = state.get("params")
    if runParams is None:
        raise FileNotFoundError(
            f"[etape 10] aucun run entraîné dans '{runDir}' (state.json sans 'params') -- "
            f"lance l'étape 9 pour '{cellType}' (finalRunTag={finalRunTag!r}) avant d'évaluer."
        )

    print(f"[etape 10] '{cellType}' <- '{runDir}' -- hyperparamètres du run : {runParams}")

    # Avertissement VISIBLE (pas une erreur) si le run chargé ne correspond plus
    # aux best_params_<arch>.json COURANTS -- cas normal si une étude Optuna plus
    # récente a tourné depuis l'entraînement de ce run (ex. runs hérités vs étude
    # en cours) : le modèle évalué n'est alors PAS celui que la dernière étude
    # désigne, et l'avertissement le rend visible.
    comparableKeys = ["tokenizationConfig", "learningRate", "hiddenDim", "embDim", "dropout", "teacherForcingDecay"]
    currentBestParams = runHyperparams[cellType]
    # gru_attention réutilise DÉLIBÉRÉMENT le fichier best_params_gru.json (cf. cellule
    # "Chargement des meilleurs hyperparamètres") -- nommer la vraie source ici.
    currentBestParamsSource = f"best_params_{runCellType[cellType]}.json"
    diffs = params_differences(
        {k: runParams[k] for k in comparableKeys if k in runParams},
        {k: currentBestParams[k] for k in comparableKeys if k in currentBestParams}
    )
    if diffs:
        print("!" * 78)
        print(
            f"[etape 10] ATTENTION : le run '{cellType}' ('{runDir}') a été entraîné avec des "
            f"hyperparamètres DIFFÉRENTS des {currentBestParamsSource} courants -- {'; '.join(diffs)}."
        )
        print("[etape 10] Le modèle évalué n'est PAS celui désigné par la dernière étude Optuna.")
        print("!" * 78)

    tokenizationConfig = runParams["tokenizationConfig"]
    resources = getFinalResources(tokenizationConfig)
    frVocab, enVocab = resources["frVocab"], resources["enVocab"]
    model = buildModel(
        len(frVocab), len(enVocab), runParams["embDim"], runParams["hiddenDim"], runCellType[cellType],
        runParams["dropout"], use_attention=runUseAttention[cellType],
    )
    bestPath = Path(runDir) / "best.pt"
    model.load_state_dict(torch.load(bestPath, map_location=device, weights_only=False))
    model.to(device)
    model.eval()
    return model, frVocab, enVocab, resources

In [ ]:
# --- Génération libre INCRÉMENTALE sur le test, par run -------------------------
hypothesesParRun: dict[str, list[str]] = {}

for cellType in finalArchitectures:
    model, frVocab, enVocab, resources = loadBestModel(cellType)
    # même max_len que celui dérivé pendant l'entraînement de ce run (cf. getFinalResources)
    testDataset = makeDataloaders({"test": testDfFinal}, frVocab, enVocab, resources["maxLen"], batchSize)["test"].dataset
    hypPath = Path(finalRunDirs[cellType]) / "hypotheses.jsonl"

    print(f"\n=== Étape 10 -- génération libre '{cellType}' sur {len(testDataset)} paires ===")
    hypotheses = generate_translations_resumable(model, testDataset, enVocab, device, hypPath, batch_size=batchSize)
    hypothesesParRun[cellType] = hypotheses
    mark_step_done(finalRunDirs[cellType], "generate", nHypotheses=len(hypotheses))

assert all(len(hypothesesParRun[c]) == len(testDfFinal) for c in finalArchitectures), (
    "chaque run doit produire exactement une hypothèse par phrase de test"
)


In [ ]:
# --- Métriques par run x bin : SacreBLEU + METEOR sur le test complet, --------
# --- BERTScore sur l'échantillon stratifié ~3000 paires (commun aux 3 runs) ---
bertscoreSampleFraction = min(
    1.0, (debugBertscoreSampleSize if debugMode else bertscoreSampleSize) / max(len(testDfFinal), 1)
)
testBertscoreSample, _diagBertscore = subsampleStratifiedByLength(
    testDfFinal, bertscoreSampleFraction, lengthBinThreshold, randomSeed
)
# `subsampleStratifiedByLength` réindexe son retour (reset_index) -- on retrouve les
# positions ORIGINALES dans `testDfFinal` via l'ensemble des cibles EN conservées
# (mêmes groupes que la fonction, alignement garanti avec hypothesesParRun/referencesFinal).
ciblesBertscore = set(testBertscoreSample["en"])
indicesBertscore = testDfFinal.index[testDfFinal["en"].isin(ciblesBertscore)].tolist()
print(f"[etape 10] échantillon stratifié BERTScore : {len(indicesBertscore)}/{len(testDfFinal)} paires.")

metricsRows = []
for cellType in finalArchitectures:
    hyps = hypothesesParRun[cellType]

    dfSansBertscore = evaluate_by_length_bin(hyps, referencesFinal, sourcesFinal, threshold=lengthBinThreshold, compute_bertscore=False)

    hypsBertscore = [hyps[i] for i in indicesBertscore]
    refsBertscore = [referencesFinal[i] for i in indicesBertscore]
    sourcesBertscoreRun = [sourcesFinal[i] for i in indicesBertscore]
    dfAvecBertscore = evaluate_by_length_bin(
        hypsBertscore, refsBertscore, sourcesBertscoreRun, threshold=lengthBinThreshold,
        compute_bertscore=True, bertscore_model_type=bertscoreModelType,
    )

    fusion = dfSansBertscore.drop(columns=["bertscorePrecision", "bertscoreRecall", "bertscoreF1"]).merge(
        dfAvecBertscore[["bin", "bertscorePrecision", "bertscoreRecall", "bertscoreF1"]], on="bin"
    )
    fusion.insert(0, "run", cellType)
    metricsRows.append(fusion)

    metricsPath = Path(finalRunDirs[cellType]) / "metrics.json"
    with open(metricsPath, "w", encoding="utf-8") as f:
        json.dump(fusion.to_dict(orient="records"), f, ensure_ascii=False, indent=2)
    mark_step_done(finalRunDirs[cellType], "metrics", metricsPath=str(metricsPath))

metricsFinalDf = pd.concat(metricsRows, ignore_index=True)
display(metricsFinalDf)


### Analyses (ce qui fait la note, Q13/Q14)

In [ ]:
# 1. Biais du teacher forcing (Q14a) -- accuracy token-à-token en teacher
# forcing pur vs en génération libre, même modèle, même dataloader (val).
tfBiasRows = []
for cellType in finalArchitectures:
    model, _frVocab, _enVocab, resources = loadBestModel(cellType)

    resultatBias = teacher_forcing_bias(model, resources["loaders"]["val"], device, pad_id=0)
    resultatBias["run"] = cellType
    tfBiasRows.append(resultatBias)

tfBiasDf = pd.DataFrame(tfBiasRows)[["run", "teacherForcingAccuracy", "freeRunningAccuracy", "gap", "nTokens"]]
display(tfBiasDf)


In [ ]:
# 2. Désaccords entre métriques -- où SacreBLEU (par phrase) pénalise une
# hypothèse que METEOR et/ou BERTScore acceptent (typiquement les synonymes).
# Calculé sur le run `disagreementsReferenceRun`, restreint à l'échantillon
# stratifié (seul sous-ensemble où un BERTScore PAR PHRASE est disponible).
hypsEch = [hypothesesParRun[disagreementsReferenceRun][i] for i in indicesBertscore]
refsEch = [referencesFinal[i] for i in indicesBertscore]
sourcesEch = [sourcesFinal[i] for i in indicesBertscore]

try:
    import bert_score as bertScoreLib

    _precisionEch, _recallEch, f1Ech = bertScoreLib.score(
        hypsEch, refsEch, model_type=bertscoreModelType, num_layers=5, device="cpu", batch_size=32, verbose=False,
    )
    bertscoreF1Liste = f1Ech.tolist()
except Exception as exc:  # dégradation volontaire -- jamais bloquer le notebook pour cette analyse
    print(f"[etape 10] BERTScore indisponible pour l'analyse des désaccords ({exc}) -- analyse basée sur METEOR seul.")
    bertscoreF1Liste = None

disagreementsDf = metric_disagreements(
    hypsEch, refsEch, sources=sourcesEch, bertscore_f1=bertscoreF1Liste,
    bleu_low=20.0, meteor_high=0.5, bertscore_high=0.85,
)
display(disagreementsDf)


In [ ]:
# 3. Exemples source / hypothèse / référence, dans les deux bins (Q14d)
samplesDf = sample_predictions(
    sourcesFinal, hypothesesParRun[disagreementsReferenceRun], referencesFinal,
    n=samplePredictionsN, seed=randomSeed, threshold=lengthBinThreshold,
)
display(samplesDf)


In [ ]:
# Courbes d'évolution par epoch (train loss, val loss, perplexité) -- les 3 runs
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for cellType in finalArchitectures:
    historique = finalResults[cellType]["history"]
    epochsRange = range(1, len(historique["trainLoss"]) + 1)
    axes[0].plot(epochsRange, historique["trainLoss"], marker="o", label=cellType)
    axes[1].plot(epochsRange, historique["valLoss"], marker="o", label=cellType)
    axes[2].plot(epochsRange, historique["valPerplexity"], marker="o", label=cellType)

axes[0].set_title("Train loss")
axes[1].set_title("Val loss")
axes[2].set_title("Val perplexité")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()

figCourbesPath = Path(runsDir) / "courbes_entrainement.png"
figCourbesPath.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(figCourbesPath, dpi=120)
plt.show()
print(f"[etape 10] courbes -> {figCourbesPath}")


## 10. Conclusion — étape 10

`metricsFinalDf`, `tfBiasDf`, `disagreementsDf` et `samplesDf` ci-dessus sont la
base de la discussion attendue par l'étape 10. Grille de lecture, **sans
présupposer le résultat** -- le plan prévient explicitement que sur ce corpus
(médiane 6-7 mots) l'écart architecture/attention est attendu **faible sur les
phrases courtes** (la littérature, Bahdanau 2015, situe la séparation des
courbes vers 15-20 mots) :

- **Bin court vs bin long** : comparer `metricsFinalDf` filtré sur
  `bin == "courte"` puis `bin == "longue"`, pour les 3 runs. Si RNN et GRU sont
  proches sur le bin court mais s'écartent sur le bin long, c'est la signature
  attendue du décrochage du RNN simple sur les séquences longues.
- **Effet de l'attention** : comparer `gru` et `gru_attention` (mêmes
  hyperparamètres, **une seule variable change**) -- l'écart doit être plus
  visible sur le bin long que sur le bin court, si la fenêtre de longueur de ce
  corpus suffit à le révéler.
- **`tfBiasDf`** : l'accuracy en teacher forcing doit être NETTEMENT supérieure
  à celle en génération libre pour les 3 runs -- c'est la démonstration
  chiffrée du biais (Q14a), pas un dysfonctionnement à corriger.
- **`disagreementsDf`** : chaque ligne est une hypothèse que SacreBLEU pénalise
  mais que METEOR (et/ou BERTScore) accepte -- typiquement un synonyme ou une
  reformulation correcte. Une table vide n'est pas un échec : cela peut aussi
  signifier que ce corpus, très normalisé (vocabulaire fermé, phrases
  courtes), laisse peu de place à la paraphrase.
- **Un résultat serré, bien expliqué, vaut un résultat positif**
  (`docs/plan-seq2seq.md`) : si l'écart RNN/GRU ou l'effet de l'attention
  restent faibles même sur le bin long, la conclusion honnête est que ce
  corpus (phrases courtes) reste sous, ou proche, du seuil de décrochage
  documenté par la littérature -- ce n'est pas un échec de l'implémentation,
  c'est une propriété du dataset.

**Ne pas préremplir ces constats avec des chiffres inventés** : à l'exécution
réelle (corpus complet, budgets de la cellule Paramètres de l'étape 9), relire
cette section à la lumière des valeurs effectivement obtenues.
